# BuzzSpot RF-DETR Large 1344 - causal six-frame temporal inference

Inference-only notebook built around the selected 1344-resolution EMA checkpoint:

```text
/content/drive/MyDrive/BuzzSpot/weights/
rfdetr_large_trainval_os_1344_32ep_selected_for_inference.pth
```

For every scored keyframe \(t\), the notebook uses only:

\[
[t-5,t-4,t-3,t-2,t-1,t]
\]

Pipeline:

1. Recover exact six-frame windows from the challenge metadata.
2. Run the 1344 RF-DETR checkpoint once per physical frame and cache outputs resumably.
3. Preserve raw query IDs and all four class scores by reproducing RF-DETR's public preprocessing and postprocessing.
4. Build class-agnostic, motion-aware backward tracklets with Hungarian matching.
5. Measure hoverfly rescueability on the leaked validation split.
6. Evaluate fixed temporal candidates:
   - Seq-NMS-style mean and bounded-max rescoring
   - noisy-OR temporal support
   - corrected isolated-proposal penalty
   - diagnostic-gated bee ↔ hoverfly class fusion
   - low-score current-proposal recovery
   - optional true box propagation
7. Rank candidates on validation and export the strongest test candidates as separate zip files.

No model training, SAHI, future frames, or learned calibration model is used.

## 1. Install the pinned inference environment

This matches the successful RF-DETR notebooks. The cell restarts the Colab runtime once after installation.

In [1]:
import os
import sys
import subprocess
from pathlib import Path

SETUP_MARKER = Path("/content/.buzzspot_rfdetr_temporal_env_ready_v1")

if not SETUP_MARKER.exists():
    print("Installing pinned temporal-inference environment...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
        "rfdetr[train,loggers]==1.8.3",
        "pycocotools",
        "tqdm",
        "supervision>=0.29.0",
        "scipy",
        "pandas",
        "matplotlib",
    ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
        "--force-reinstall", "Pillow==12.3.0",
    ])
    SETUP_MARKER.write_text("ready")
    print("Installed. Restarting the runtime once.")
    print("After reconnecting, run the notebook again from the top.")
    os.kill(os.getpid(), 9)
else:
    print("Pinned environment already installed.")

Pinned environment already installed.


## 1.1 Import smoke test

In [2]:
import json
import math
import os
import re
import gc
import gzip
import pickle
import random
import shutil
import tarfile
import tempfile
import zipfile
from collections import Counter, defaultdict
from contextlib import redirect_stdout
from dataclasses import dataclass
from io import StringIO
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd
import torch
import torchvision.transforms.functional as TVF
from PIL import Image, ImageDraw
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm

import PIL
import rfdetr
import supervision as sv
from rfdetr import RFDETRLarge

print("Pillow:", PIL.__version__)
print("RF-DETR:", getattr(rfdetr, "__version__", "unknown"))
print("Supervision:", sv.__version__)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Pillow: 12.3.0
RF-DETR: unknown
Supervision: 0.29.1
Torch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


/usr/local/lib/python3.12/dist-packages/rfdetr/models/weights.py:258: FutureWarning: target=True is deprecated since `v0.8`; use `TargetMode.ARGS_REMAP` instead. Will be removed in `v1.0`.
  @deprecated(target=True, args_mapping={"train_config": None}, deprecated_in="1.7.0", remove_in="1.9.0", num_warns=-1)


## 2. Mount Drive and locate the challenge archives

In [3]:
from google.colab import drive
drive.mount("/content/drive")

import glob

zip_hits = glob.glob("/content/drive/MyDrive/**/BuzzSet_challenge.zip", recursive=True)
assert zip_hits, "BuzzSet_challenge.zip not found under MyDrive"
ZIP_PATH = Path(zip_hits[0])

tar_hits = glob.glob(
    "/content/drive/MyDrive/**/19557529600-BuzzSet_challenge_testphase.tar",
    recursive=True,
)
if not tar_hits:
    tar_hits = glob.glob(
        "/content/drive/MyDrive/**/*BuzzSet_challenge_testphase*.tar*",
        recursive=True,
    )
assert tar_hits, "BuzzSet challenge test-phase tar not found under MyDrive"
TAR_PATH = Path(tar_hits[0])

print("Train/valid zip:", ZIP_PATH)
print("Test-phase tar:", TAR_PATH)

Mounted at /content/drive
Train/valid zip: /content/drive/MyDrive/Colab Notebooks/CVPPA ECCV 26 BuzzSpot Challenge/BuzzSet_challenge.zip
Test-phase tar: /content/drive/MyDrive/Colab Notebooks/CVPPA ECCV 26 BuzzSpot Challenge/19557529600-BuzzSet_challenge_testphase.tar


## 3. Configuration

The defaults implement the locked plan. Inference and association caches are sharded and resumable, so rerunning the notebook does not repeat completed work.

In [4]:
# -----------------------------
# Model and submission contract
# -----------------------------
CLASS_NAMES = ["bee", "bumblebee", "hoverfly", "moth"]
NUM_CLASSES = len(CLASS_NAMES)
CATEGORY_IDS = {name: i + 1 for i, name in enumerate(CLASS_NAMES)}

RESOLUTION = 1344
SOURCE_WEIGHTS = Path(
    "/content/drive/MyDrive/BuzzSpot/weights/"
    "rfdetr_large_trainval_os_1344_32ep_selected_for_inference.pth"
)
assert SOURCE_WEIGHTS.exists(), f"Missing selected 1344 EMA weights: {SOURCE_WEIGHTS}"

W_FULL, H_FULL = 1920, 1080

# RF-DETR public postprocessing selects at most 300 query/class pairs.
RAW_PROPOSAL_THRESHOLD = 0.001
BASELINE_SUBMISSION_THRESHOLD = 0.01
POSTPROCESS_NUM_SELECT = 300
TRACK_MAX_PROPOSALS = 200
FINAL_MAX_DETECTIONS = 100
INFERENCE_BATCH_SIZE = 2

# Expected leaked-validation result from the successful 1344 checkpoint.
EXPECTED_BASELINE_MAP = 0.8864
BASELINE_MAP_TOLERANCE = 0.02

# -----------------------------
# Temporal evidence
# -----------------------------
RHO = 0.8
LAMBDA_VALUES = [0.25, 0.35, 0.45]
DELTA_VALUES = [0.0, 0.10, 0.15]
DEFAULT_LAMBDA = 0.35
DEFAULT_DELTA = 0.15
BOUNDED_MAX_BLEND = 0.50
ALPHA_MAX = 0.50
EPS = 1e-8

# Association weights and gates.
ASSOC = {
    "w_center": 0.55,
    "w_area": 0.20,
    "w_aspect": 0.10,
    "w_iou": 0.10,
    "w_motion": 0.05,
    "class_mismatch_penalty": 0.03,
    "max_center_norm": 8.0,
    "max_log_area": 1.8,
    "max_log_aspect": 1.5,
    "max_motion_norm": 8.0,
    "max_cost": 5.0,
    "min_scale_px": 8.0,
    "max_gap": 1,
}

# Class-correction gates.
MIN_CLASS_MATCHES = 2
MIN_CLASS_TRACK_RELIABILITY = 0.15
MIN_CLASS_SUPPORT_MARGIN = 0.08
MIN_RELIABLE_MATCH = 0.20
MIN_RESCUEABLE_FRACTION = 0.08
ALLOW_ONLY_BEE_HOVERFLY_RELABEL = True

# True propagation is a separate aggressive ablation.
ENABLE_TRUE_PROPAGATION = True
PROP_MIN_ANCHOR_SCORE = 0.03
PROP_MIN_MATCHES = 2
PROP_MIN_TRACK_RELIABILITY = 0.25
PROP_SCORE_DECAY = 0.60
PROP_DUP_IOU = 0.10
PROP_DUP_CENTER_NORM = 2.0

# -----------------------------
# Execution controls
# -----------------------------
RUN_VALID_INFERENCE_CACHE = True
RUN_TEST_INFERENCE_CACHE = True
RUN_VALID_TRACK_CACHE = True
RUN_TEST_TRACK_CACHE = True
RUN_HOVERFLY_DIAGNOSTIC = True
RUN_VALID_CANDIDATE_SWEEP = True
RUN_TEST_EXPORT = True

CACHE_SHARD_SIZE = 250
TRACK_SHARD_SIZE = 150
AUTO_EXPORT_TOP_K = 3
RANDOM_SEED = 0
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# -----------------------------
# Paths
# -----------------------------
PROJECT_DIR = Path("/content/drive/MyDrive/BuzzSpot")
TEMPORAL_DIR = PROJECT_DIR / "temporal_rfdetr_1344"
CACHE_DIR = TEMPORAL_DIR / "inference_cache"
TRACK_DIR = TEMPORAL_DIR / "track_cache"
REPORT_DIR = TEMPORAL_DIR / "reports"
DEBUG_DIR = TEMPORAL_DIR / "debug_visuals"
SUBMISSIONS_DIR = PROJECT_DIR / "submissions"
LOCAL_FRAME_ROOT = Path("/content/buzzspot_temporal_frames")

for directory in [
    TEMPORAL_DIR,
    CACHE_DIR,
    TRACK_DIR,
    REPORT_DIR,
    DEBUG_DIR,
    SUBMISSIONS_DIR,
    LOCAL_FRAME_ROOT,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Source weights:", SOURCE_WEIGHTS)
print("Temporal workspace:", TEMPORAL_DIR)
print("Resolution:", RESOLUTION)
print("Raw proposal threshold:", RAW_PROPOSAL_THRESHOLD)
print("Final threshold:", BASELINE_SUBMISSION_THRESHOLD)
print("Track proposal cap:", TRACK_MAX_PROPOSALS)

Source weights: /content/drive/MyDrive/BuzzSpot/weights/rfdetr_large_trainval_os_1344_32ep_selected_for_inference.pth
Temporal workspace: /content/drive/MyDrive/BuzzSpot/temporal_rfdetr_1344
Resolution: 1344
Raw proposal threshold: 0.001
Final threshold: 0.01
Track proposal cap: 200


## 4. Load challenge metadata and construct exact causal windows

The builder prefers explicit previous-frame references when the annotation schema provides them. Otherwise it groups frames by sequence/video metadata and sorts by frame index or the numeric filename suffix.

Validation targets are annotated keyframes. Test targets are images whose `is_keyframe` flag is true.

In [5]:
zf = zipfile.ZipFile(ZIP_PATH)
tf = tarfile.open(TAR_PATH, "r:*")

zip_names = set(zf.namelist())
tar_members = {m.name: m for m in tf.getmembers() if m.isfile()}
tar_names = set(tar_members)

def find_zip(rel):
    rel = str(rel).lstrip("/")
    candidates = [rel, f"BuzzSet_challenge/{rel}"]
    for candidate in candidates:
        if candidate in zip_names:
            return candidate
    suffix = "/" + rel
    hits = [name for name in zip_names if name.endswith(suffix)]
    return hits[0] if hits else None

def find_tar(rel):
    rel = str(rel).lstrip("/")
    candidates = [
        rel,
        f"BuzzSet_challenge/{rel}",
        f"BuzzSet_challenge_testphase/{rel}",
    ]
    for candidate in candidates:
        if candidate in tar_names:
            return candidate
    suffix = "/" + rel
    hits = [name for name in tar_names if name.endswith(suffix)]
    return hits[0] if hits else None

def load_tar_json(filename):
    member_name = find_tar(f"annotations/{filename}") or find_tar(filename)
    assert member_name is not None, f"{filename} not found in tar"
    with tf.extractfile(tar_members[member_name]) as handle:
        obj = json.load(handle)
    print("Loaded:", filename, "from", member_name)
    return obj

train_json = load_tar_json("train.json")
valid_json = load_tar_json("valid.json")
test_json = load_tar_json("test_testphase.json")

print("Validation image keys:", sorted(valid_json["images"][0].keys()))
print("Test image keys:", sorted(test_json["images"][0].keys()))
print("Validation sample:", valid_json["images"][0])
print("Test sample:", test_json["images"][0])

Loaded: train.json from BuzzSpot_testphase/annotations/train.json
Loaded: valid.json from BuzzSpot_testphase/annotations/valid.json
Loaded: test_testphase.json from BuzzSpot_testphase/annotations/test_testphase.json
Validation image keys: ['file_name', 'frame_index', 'height', 'id', 'is_keyframe', 'video_id', 'width']
Test image keys: ['file_name', 'frame_index', 'height', 'id', 'is_keyframe', 'video_id', 'width']
Validation sample: {'id': 1, 'video_id': 1, 'frame_index': 33205, 'is_keyframe': False, 'file_name': 'valid_video_1/valid_video_1_033205.jpg', 'width': 1920, 'height': 1080}
Test sample: {'id': 1, 'video_id': 1, 'frame_index': 5725, 'is_keyframe': False, 'file_name': 'test_video_1/test_video_1_005725.jpg', 'width': 1920, 'height': 1080}


In [6]:
FRAME_INDEX_KEYS = [
    "frame_index",
    "frame_idx",
    "frame_id",
    "frame_number",
    "frame_no",
    "frame",
    "index",
]
GROUP_KEYS = [
    "video_id",
    "sequence_id",
    "seq_id",
    "clip_id",
    "video_name",
    "source_video",
    "sequence",
    "clip",
    "day",
]
PREVIOUS_KEYS = [
    "previous_image_ids",
    "previous_frame_ids",
    "prev_image_ids",
    "prev_frame_ids",
    "context_image_ids",
    "preceding_image_ids",
    "past_image_ids",
    "previous_frames",
    "context_frames",
]

def truthy(value):
    if value is True or value == 1:
        return True
    if isinstance(value, str):
        return value.strip().lower() in {"1", "true", "yes", "y"}
    return False

def frame_group_key(image_record):
    metadata_parts = []
    for key in GROUP_KEYS:
        value = image_record.get(key)
        if value not in (None, ""):
            metadata_parts.append(f"{key}:{value}")

    # Combine available metadata instead of trusting one coarse field such as day.
    if metadata_parts:
        return "|".join(metadata_parts)

    file_path = PurePosixPath(str(image_record["file_name"]))
    parent = str(file_path.parent)
    if parent not in {"", "."}:
        return f"parent:{parent}"

    # Flat filenames often encode a video/clip prefix followed by a frame number.
    stem = file_path.stem
    numeric_matches = list(re.finditer(r"\d+", stem))
    if numeric_matches:
        prefix = stem[:numeric_matches[-1].start()].rstrip("_-. ")
        if prefix:
            return f"stem_prefix:{prefix}"

    return f"parent:{parent}"

def frame_index_value(image_record):
    for key in FRAME_INDEX_KEYS:
        value = image_record.get(key)
        if value is None:
            continue
        try:
            return int(value)
        except (TypeError, ValueError):
            pass

    stem = PurePosixPath(str(image_record["file_name"])).stem
    numbers = re.findall(r"\d+", stem)
    if numbers:
        return int(numbers[-1])

    return int(image_record["id"])

def normalize_filename(name):
    return str(PurePosixPath(str(name))).lstrip("./")

def resolve_explicit_previous(image_record, by_id, by_filename):
    raw = None
    for key in PREVIOUS_KEYS:
        if key in image_record and image_record[key] not in (None, []):
            raw = image_record[key]
            break
    if raw is None:
        return None

    if isinstance(raw, (str, int, dict)):
        raw = [raw]

    resolved = []
    for item in raw:
        if isinstance(item, dict):
            item = item.get("id", item.get("image_id", item.get("file_name")))

        record = None
        try:
            record = by_id.get(int(item))
        except (TypeError, ValueError):
            record = None

        if record is None and item is not None:
            record = by_filename.get(normalize_filename(item))

        if record is not None:
            resolved.append(record)

    if not resolved:
        return None

    resolved = sorted(
        {int(record["id"]): record for record in resolved}.values(),
        key=lambda record: (frame_index_value(record), int(record["id"])),
    )
    return resolved[-5:]

def build_sequence_windows(coco, split_name, target_ids):
    images = [dict(image) for image in coco["images"]]
    by_id = {int(image["id"]): image for image in images}
    by_filename = {
        normalize_filename(image["file_name"]): image
        for image in images
    }

    grouped = defaultdict(list)
    for image in images:
        grouped[frame_group_key(image)].append(image)

    positions = {}
    for group, records in grouped.items():
        records.sort(key=lambda record: (frame_index_value(record), int(record["id"])))
        for position, record in enumerate(records):
            positions[int(record["id"])] = (group, position, records)

    windows = {}
    failures = []

    for target_id in sorted(map(int, target_ids)):
        target = by_id[target_id]
        explicit_previous = resolve_explicit_previous(target, by_id, by_filename)

        if explicit_previous is not None and len(explicit_previous) >= 5:
            previous = explicit_previous[-5:]
        else:
            group, position, records = positions[target_id]
            previous = records[max(0, position - 5):position]

        if len(previous) != 5:
            failures.append({
                "image_id": target_id,
                "file_name": target["file_name"],
                "group": frame_group_key(target),
                "previous_found": len(previous),
            })
            continue

        ordered = previous + [target]
        ordered_ids = [int(record["id"]) for record in ordered]

        # Hard causal and group checks.
        assert ordered_ids[-1] == target_id
        assert len(set(ordered_ids)) == 6
        assert len({frame_group_key(record) for record in ordered}) == 1

        windows[target_id] = {
            "split": split_name,
            "group": frame_group_key(target),
            "keyframe_id": target_id,
            "frame_ids": ordered_ids,
            "frame_indices": [frame_index_value(record) for record in ordered],
            "file_names": [record["file_name"] for record in ordered],
        }

    if failures:
        failure_path = REPORT_DIR / f"{split_name}_sequence_failures.json"
        failure_path.write_text(json.dumps(failures, indent=2))
        raise RuntimeError(
            f"{len(failures)} {split_name} keyframes did not have exactly five "
            f"previous frames. Inspect {failure_path}"
        )

    return windows, by_id

valid_annotated_ids = {
    int(annotation["image_id"])
    for annotation in valid_json.get("annotations", [])
}

test_keyframe_ids = {
    int(image["id"])
    for image in test_json["images"]
    if truthy(image.get("is_keyframe"))
}
assert test_keyframe_ids, "No test keyframes found from is_keyframe."

valid_windows, valid_image_records = build_sequence_windows(
    valid_json,
    "valid",
    valid_annotated_ids,
)
test_windows, test_image_records = build_sequence_windows(
    test_json,
    "test",
    test_keyframe_ids,
)

def windows_to_dataframe(windows):
    rows = []
    for keyframe_id, window in windows.items():
        row = {
            "keyframe_id": keyframe_id,
            "group": window["group"],
        }
        # frame_ids are oldest -> current.
        for position, image_id in enumerate(window["frame_ids"]):
            lag = 5 - position
            row[f"t_minus_{lag}" if lag else "t"] = image_id
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["group", "keyframe_id"])

valid_window_df = windows_to_dataframe(valid_windows)
test_window_df = windows_to_dataframe(test_windows)

valid_window_df.to_csv(REPORT_DIR / "valid_temporal_windows.csv", index=False)
test_window_df.to_csv(REPORT_DIR / "test_temporal_windows.csv", index=False)

print("Validation keyframes:", len(valid_windows))
print("Test keyframes:", len(test_windows))
display(valid_window_df.head())
display(test_window_df.head())

Validation keyframes: 932
Test keyframes: 4763


,keyframe_id,group,t_minus_5,t_minus_4,t_minus_3,t_minus_2,t_minus_1,t
0,6,video_id:1,1,2,3,4,5,6
337,2046,video_id:10,2041,2042,2043,2044,2045,2046
338,2052,video_id:10,2047,2048,2049,2050,2051,2052
339,2058,video_id:10,2053,2054,2055,2056,2057,2058
340,2064,video_id:10,2059,2060,2061,2062,2063,2064


,keyframe_id,group,t_minus_5,t_minus_4,t_minus_3,t_minus_2,t_minus_1,t
0,6,video_id:1,1,2,3,4,5,6
1,12,video_id:1,7,8,9,10,11,12
2,18,video_id:1,13,14,15,16,17,18
3,24,video_id:1,19,20,21,22,23,24
4,30,video_id:1,25,26,27,28,29,30


## 5. Extract only the physical frames required by the six-frame windows

All unique frames are extracted once. The operation is idempotent.

In [7]:
def flat_name(value):
    return str(value).replace("/", "__")

def extract_required_frames(split_name, windows, image_records):
    required_ids = sorted({
        int(image_id)
        for window in windows.values()
        for image_id in window["frame_ids"]
    })

    output_dir = LOCAL_FRAME_ROOT / split_name
    output_dir.mkdir(parents=True, exist_ok=True)
    image_id_to_path = {}

    for image_id in tqdm(required_ids, desc=f"extract {split_name} temporal frames"):
        image = image_records[image_id]
        output_name = (
            f"{image_id:010d}__{flat_name(image['file_name'])}"
        )
        destination = output_dir / output_name

        if not destination.exists() or destination.stat().st_size == 0:
            if split_name == "valid":
                member_name = (
                    find_zip(f"valid/{image['file_name']}")
                    or find_zip(image["file_name"])
                )
                assert member_name is not None, (
                    f"Missing valid frame in zip: {image['file_name']}"
                )
                with zf.open(member_name) as source, open(destination, "wb") as target:
                    shutil.copyfileobj(source, target)
            elif split_name == "test":
                member_name = (
                    find_tar(f"test_testphase/{image['file_name']}")
                    or find_tar(image["file_name"])
                )
                assert member_name is not None, (
                    f"Missing test frame in tar: {image['file_name']}"
                )
                with tf.extractfile(tar_members[member_name]) as source, open(destination, "wb") as target:
                    shutil.copyfileobj(source, target)
            else:
                raise ValueError(split_name)

        image_id_to_path[image_id] = destination

    return image_id_to_path

valid_frame_paths = extract_required_frames(
    "valid",
    valid_windows,
    valid_image_records,
)
test_frame_paths = extract_required_frames(
    "test",
    test_windows,
    test_image_records,
)

print("Unique validation frames:", len(valid_frame_paths))
print("Unique test frames:", len(test_frame_paths))

extract valid temporal frames:   0%|          | 0/5592 [00:00<?, ?it/s]

extract test temporal frames:   0%|          | 0/28578 [00:00<?, ?it/s]

Unique validation frames: 5592
Unique test frames: 28578


## 6. Build the validation ground-truth file used by both evaluators

In [8]:
valid_target_ids = set(valid_windows)

valid_gt = {
    "info": {
        "description": (
            "BuzzSpot validation annotated keyframes for temporal diagnostics; "
            "the detector was trained on these images."
        )
    },
    "licenses": valid_json.get("licenses", []),
    "categories": [
        {"id": i + 1, "name": name, "supercategory": "pollinator"}
        for i, name in enumerate(CLASS_NAMES)
    ],
    "images": [
        {
            "id": int(image["id"]),
            "file_name": image["file_name"],
            "width": int(image.get("width", W_FULL)),
            "height": int(image.get("height", H_FULL)),
        }
        for image in valid_json["images"]
        if int(image["id"]) in valid_target_ids
    ],
    "annotations": [],
}

next_annotation_id = 1
for annotation in valid_json.get("annotations", []):
    if int(annotation["image_id"]) not in valid_target_ids:
        continue
    copied = dict(annotation)
    copied["id"] = int(copied.get("id", next_annotation_id))
    copied["image_id"] = int(copied["image_id"])
    copied["category_id"] = int(copied["category_id"])
    copied["bbox"] = [float(value) for value in copied["bbox"]]
    copied["area"] = float(
        copied.get("area", copied["bbox"][2] * copied["bbox"][3])
    )
    copied["iscrowd"] = int(copied.get("iscrowd", 0))
    valid_gt["annotations"].append(copied)
    next_annotation_id = max(next_annotation_id, copied["id"] + 1)

VALID_GT_PATH = Path("/content/buzzspot_temporal_valid_gt.json")
VALID_GT_PATH.write_text(json.dumps(valid_gt))

print("Validation GT images:", len(valid_gt["images"]))
print("Validation GT boxes:", len(valid_gt["annotations"]))
print("Validation GT:", VALID_GT_PATH)

Validation GT images: 932
Validation GT boxes: 1116
Validation GT: /content/buzzspot_temporal_valid_gt.json


## 7. Load the selected 1344 EMA checkpoint

The class-name order and four-class head are asserted before any expensive inference.

In [9]:
model = RFDETRLarge.from_checkpoint(str(SOURCE_WEIGHTS))

print("Loaded:", SOURCE_WEIGHTS)
print("Model resolution:", model.model_config.resolution)
print("Underlying resolution:", model.model.resolution)
print("Class names:", model.class_names)

assert model.model_config.resolution == RESOLUTION
assert model.model.resolution == RESOLUTION
assert int(model.model_config.num_classes) == NUM_CLASSES

normalized_names = [
    str(name).strip().lower().replace("_", "").replace("-", "").replace(" ", "")
    for name in model.class_names
]
expected_names = [
    name.replace("_", "").replace("-", "").replace(" ", "")
    for name in CLASS_NAMES
]
assert normalized_names == expected_names, (
    f"Unexpected class mapping: {model.class_names}"
)

smoke_path = next(iter(valid_frame_paths.values()))
smoke_detections = model.predict(
    str(smoke_path),
    threshold=0.5,
    include_source_image=False,
)
print("Public predict smoke detections:", len(smoke_detections))
print("Postprocess num_select:", model.model.postprocess.num_select)

# Public predict moves the lazily loaded model to its target device.
model.model.model.eval()
DEVICE = model.model.device
print("Inference device:", DEVICE)

gc.collect()
torch.cuda.empty_cache()

[2026-07-07 05:18:27] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-07-07 05:18:27] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-07-07 05:18:28] [WARNING] rf-detr - load_pretrain_weights: args.num_queries absent; inferred ckpt_num_queries=300 from tensor rows 3900 ÷ ckpt_group_detr=13.


Loaded: /content/drive/MyDrive/BuzzSpot/weights/rfdetr_large_trainval_os_1344_32ep_selected_for_inference.pth
Model resolution: 1344
Underlying resolution: 1344
Class names: ['bee', 'bumblebee', 'hoverfly', 'moth']


[2026-07-07 05:18:28] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. For full GPU throughput (e.g. ~8x on T4 via FP16 Tensor Cores), call model.optimize_for_inference(dtype=torch.float16).


Public predict smoke detections: 1
Postprocess num_select: 300
Inference device: cuda


## 8. Raw-query inference with an exact public-API parity check

RF-DETR's public `predict()` method flattens query/class pairs and returns only the selected class and confidence. The temporal pipeline needs stable query IDs and all four foreground class scores.

This cell reproduces RF-DETR 1.8.3's own preprocessing:

- RGB conversion
- tensor conversion
- square resize to 1344
- ImageNet normalization
- raw model forward
- sigmoid class probabilities
- top-300 query/class-pair selection
- source-image box scaling

If the parity check fails, the notebook automatically falls back to public `predict()` outputs. The fallback still supports association and reranking, but disables full probability fusion.

In [10]:
def cxcywh_normalized_to_xyxy(boxes, width, height):
    boxes = np.asarray(boxes, dtype=np.float32)
    cx, cy, bw, bh = boxes.T
    x1 = (cx - bw / 2.0) * width
    y1 = (cy - bh / 2.0) * height
    x2 = (cx + bw / 2.0) * width
    y2 = (cy + bh / 2.0) * height
    result = np.stack([x1, y1, x2, y2], axis=1)
    result[:, [0, 2]] = np.clip(result[:, [0, 2]], 0, width)
    result[:, [1, 3]] = np.clip(result[:, [1, 3]], 0, height)
    return result.astype(np.float32)

@torch.inference_mode()
def raw_query_infer_batch(paths):
    pil_images = []
    tensors = []
    original_sizes = []

    for path in paths:
        image = Image.open(path).convert("RGB")
        pil_images.append(image)
        width, height = image.size
        original_sizes.append((height, width))
        tensor = TVF.to_tensor(image).to(DEVICE)
        tensor = TVF.resize(tensor, [RESOLUTION, RESOLUTION])
        tensor = TVF.normalize(tensor, model.means, model.stds)
        tensors.append(tensor)

    batch = torch.stack(tensors)
    outputs = model.model.model(batch)

    if isinstance(outputs, tuple):
        outputs = {
            "pred_boxes": outputs[0],
            "pred_logits": outputs[1],
        }

    logits_batch = outputs["pred_logits"].float().cpu()
    boxes_batch = outputs["pred_boxes"].float().cpu()

    results = []

    for batch_index, (height, width) in enumerate(original_sizes):
        logits = logits_batch[batch_index]
        normalized_boxes = boxes_batch[batch_index]
        assert logits.ndim == 2
        assert logits.shape[1] >= NUM_CLASSES + 1, (
            f"Expected at least {NUM_CLASSES + 1} logit slots "
            f"(four foreground classes plus no-object), got {tuple(logits.shape)}"
        )
        all_probabilities = logits.sigmoid().numpy()
        foreground_probabilities = all_probabilities[:, :NUM_CLASSES]

        query_scores = foreground_probabilities.max(axis=1)
        query_labels = foreground_probabilities.argmax(axis=1).astype(np.int16)
        query_ids = np.arange(len(query_scores), dtype=np.int32)
        pixel_boxes = cxcywh_normalized_to_xyxy(
            normalized_boxes.numpy(),
            width,
            height,
        )

        # Reproduce RF-DETR PostProcess._select_topk over all logit slots,
        # including the no-object/background slot.
        flat_probabilities = all_probabilities.reshape(-1)
        top_count = min(
            int(model.model.postprocess.num_select),
            len(flat_probabilities),
            POSTPROCESS_NUM_SELECT,
        )
        if top_count:
            top_indices = np.argpartition(
                -flat_probabilities,
                top_count - 1,
            )[:top_count]
            top_indices = top_indices[
                np.argsort(-flat_probabilities[top_indices])
            ]
        else:
            top_indices = np.empty(0, dtype=np.int64)

        total_logit_slots = all_probabilities.shape[1]
        pair_query_ids = (top_indices // total_logit_slots).astype(np.int32)
        pair_labels = (top_indices % total_logit_slots).astype(np.int16)
        pair_scores = flat_probabilities[top_indices].astype(np.float32)

        keep_pairs = (
            (pair_scores > RAW_PROPOSAL_THRESHOLD)
            & (pair_labels >= 0)
            & (pair_labels < NUM_CLASSES)
        )
        pair_query_ids = pair_query_ids[keep_pairs]
        pair_labels = pair_labels[keep_pairs]
        pair_scores = pair_scores[keep_pairs]
        pair_boxes = pixel_boxes[pair_query_ids]

        # Tracking proposals are unique model queries, ranked by best foreground
        # score. This removes same-query class duplicates from association.
        proposal_order = np.argsort(-query_scores)
        proposal_order = proposal_order[
            query_scores[proposal_order] > RAW_PROPOSAL_THRESHOLD
        ]
        proposal_order = proposal_order[:TRACK_MAX_PROPOSALS]

        results.append({
            "width": int(width),
            "height": int(height),
            "proposal_query_ids": query_ids[proposal_order],
            "proposal_boxes": pixel_boxes[proposal_order],
            "proposal_scores": query_scores[proposal_order].astype(np.float32),
            "proposal_labels": query_labels[proposal_order],
            "proposal_class_scores": foreground_probabilities[proposal_order].astype(np.float32),
            "pair_query_ids": pair_query_ids,
            "pair_boxes": pair_boxes.astype(np.float32),
            "pair_scores": pair_scores,
            "pair_labels": pair_labels,
            "has_full_class_scores": True,
        })

    return results

def public_predict_infer_batch(paths):
    detections_list = model.predict(
        [str(path) for path in paths],
        threshold=RAW_PROPOSAL_THRESHOLD,
        include_source_image=False,
    )
    if not isinstance(detections_list, list):
        detections_list = [detections_list]

    results = []
    for path, detections in zip(paths, detections_list):
        with Image.open(path) as image:
            width, height = image.size

        boxes = np.asarray(detections.xyxy, dtype=np.float32)
        scores = np.asarray(detections.confidence, dtype=np.float32)
        labels = np.asarray(detections.class_id, dtype=np.int16)

        names = np.asarray(
            detections.data.get("class_name", []),
            dtype=object,
        )
        recognized = np.array([
            str(name).strip().lower() in CLASS_NAMES
            for name in names
        ], dtype=bool)

        boxes = boxes[recognized]
        scores = scores[recognized]
        labels = labels[recognized]

        order = np.argsort(-scores)
        boxes = boxes[order]
        scores = scores[order]
        labels = labels[order]

        proposal_count = min(len(scores), TRACK_MAX_PROPOSALS)
        class_scores = np.zeros((proposal_count, NUM_CLASSES), dtype=np.float32)
        if proposal_count:
            class_scores[
                np.arange(proposal_count),
                labels[:proposal_count],
            ] = scores[:proposal_count]

        results.append({
            "width": int(width),
            "height": int(height),
            "proposal_query_ids": np.arange(proposal_count, dtype=np.int32),
            "proposal_boxes": boxes[:proposal_count],
            "proposal_scores": scores[:proposal_count],
            "proposal_labels": labels[:proposal_count],
            "proposal_class_scores": class_scores,
            "pair_query_ids": np.arange(len(scores), dtype=np.int32),
            "pair_boxes": boxes,
            "pair_scores": scores,
            "pair_labels": labels,
            "has_full_class_scores": False,
        })
    return results

def parity_check_raw_inference(path):
    raw = raw_query_infer_batch([path])[0]
    public = public_predict_infer_batch([path])[0]

    raw_count = min(20, len(raw["pair_scores"]))
    public_count = min(20, len(public["pair_scores"]))
    count = min(raw_count, public_count)

    if count == 0:
        print("Parity image produced no low-threshold detections; raw path still completed.")
        return True

    score_error = float(np.max(np.abs(
        raw["pair_scores"][:count] - public["pair_scores"][:count]
    )))
    box_error = float(np.max(np.abs(
        raw["pair_boxes"][:count] - public["pair_boxes"][:count]
    )))
    label_equal = bool(np.array_equal(
        raw["pair_labels"][:count],
        public["pair_labels"][:count],
    ))

    print("Raw/public parity:")
    print("  compared detections:", count)
    print("  max score error:", score_error)
    print("  max box-coordinate error:", box_error)
    print("  labels identical:", label_equal)

    return score_error < 1e-4 and box_error < 1.0 and label_equal

try:
    RAW_INFERENCE_OK = parity_check_raw_inference(smoke_path)
except Exception as error:
    RAW_INFERENCE_OK = False
    print("Raw-query path failed:", repr(error))

INFERENCE_FUNCTION = (
    raw_query_infer_batch
    if RAW_INFERENCE_OK
    else public_predict_infer_batch
)

print("RAW_INFERENCE_OK:", RAW_INFERENCE_OK)
print(
    "Full four-class query probabilities:",
    "available" if RAW_INFERENCE_OK else "not available; fallback rules only",
)

Raw/public parity:
  compared detections: 20
  max score error: 9.313225746154785e-10
  max box-coordinate error: 0.0
  labels identical: True
RAW_INFERENCE_OK: True
Full four-class query probabilities: available


## 9. Resumable sharded inference cache

Each unique physical frame is inferred once. Completed shards are reused on reruns.

In [11]:
def atomic_gzip_pickle_dump(value, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(
        suffix=".pkl.gz",
        delete=False,
        dir="/content",
    ) as temporary:
        temporary_path = Path(temporary.name)

    try:
        with gzip.open(temporary_path, "wb") as handle:
            pickle.dump(value, handle, protocol=pickle.HIGHEST_PROTOCOL)
        shutil.move(str(temporary_path), str(destination))
    finally:
        temporary_path.unlink(missing_ok=True)

def gzip_pickle_load(path):
    with gzip.open(path, "rb") as handle:
        return pickle.load(handle)

def cache_shard_paths(split_name):
    directory = CACHE_DIR / split_name
    directory.mkdir(parents=True, exist_ok=True)
    return sorted(directory.glob("cache_*.pkl.gz"))

def load_inference_cache(split_name):
    merged = {}
    for shard_path in tqdm(
        cache_shard_paths(split_name),
        desc=f"load {split_name} inference shards",
    ):
        shard = gzip_pickle_load(shard_path)
        merged.update({int(key): value for key, value in shard.items()})
    return merged

def run_inference_cache(split_name, frame_paths):
    output_dir = CACHE_DIR / split_name
    output_dir.mkdir(parents=True, exist_ok=True)

    existing = load_inference_cache(split_name)
    processed_ids = set(existing)
    del existing
    gc.collect()

    pending_ids = [
        image_id
        for image_id in sorted(frame_paths)
        if image_id not in processed_ids
    ]

    print(
        f"{split_name}: {len(processed_ids)} cached, "
        f"{len(pending_ids)} pending."
    )

    next_shard_index = len(cache_shard_paths(split_name))
    shard = {}

    for start in tqdm(
        range(0, len(pending_ids), INFERENCE_BATCH_SIZE),
        desc=f"{split_name} RF-DETR temporal inference",
    ):
        batch_ids = pending_ids[start:start + INFERENCE_BATCH_SIZE]
        batch_paths = [frame_paths[image_id] for image_id in batch_ids]
        batch_results = INFERENCE_FUNCTION(batch_paths)

        for image_id, result in zip(batch_ids, batch_results):
            result["image_id"] = int(image_id)
            shard[int(image_id)] = result

        if len(shard) >= CACHE_SHARD_SIZE:
            destination = (
                output_dir / f"cache_{next_shard_index:05d}.pkl.gz"
            )
            atomic_gzip_pickle_dump(shard, destination)
            next_shard_index += 1
            shard = {}
            gc.collect()
            torch.cuda.empty_cache()

    if shard:
        destination = output_dir / f"cache_{next_shard_index:05d}.pkl.gz"
        atomic_gzip_pickle_dump(shard, destination)

    final_cache = load_inference_cache(split_name)
    assert set(final_cache) == set(frame_paths), (
        f"{split_name} cache IDs do not match required frame IDs."
    )
    print(f"{split_name} cache complete:", len(final_cache))
    return final_cache

if RUN_VALID_INFERENCE_CACHE:
    valid_cache = run_inference_cache("valid", valid_frame_paths)
else:
    valid_cache = load_inference_cache("valid")

if RUN_TEST_INFERENCE_CACHE:
    test_cache = run_inference_cache("test", test_frame_paths)
else:
    test_cache = load_inference_cache("test")

print("Validation cache frames:", len(valid_cache))
print("Test cache frames:", len(test_cache))

load valid inference shards: 0it [00:00, ?it/s]

valid: 0 cached, 5592 pending.


valid RF-DETR temporal inference:   0%|          | 0/2796 [00:00<?, ?it/s]

load valid inference shards:   0%|          | 0/23 [00:00<?, ?it/s]

valid cache complete: 5592


load test inference shards: 0it [00:00, ?it/s]

test: 0 cached, 28578 pending.


test RF-DETR temporal inference:   0%|          | 0/14289 [00:00<?, ?it/s]

load test inference shards:   0%|          | 0/115 [00:00<?, ?it/s]

test cache complete: 28578
Validation cache frames: 5592
Test cache frames: 28578


## 10. Baseline reconstruction and evaluation

The raw-query path should reproduce the successful 1344 `conf=0.01`, top-100 validation result. A hard tolerance check catches preprocessing, class-order, or postprocessing drift before temporal experiments begin.

In [12]:
def clip_xyxy(box, width, height):
    x1, y1, x2, y2 = map(float, box)
    x1 = min(max(x1, 0.0), width - 1)
    y1 = min(max(y1, 0.0), height - 1)
    x2 = min(max(x2, 0.0), width)
    y2 = min(max(y2, 0.0), height)
    if x2 - x1 < 1.0 or y2 - y1 < 1.0:
        return None
    return x1, y1, x2, y2

def cap_records(records, threshold=BASELINE_SUBMISSION_THRESHOLD):
    grouped = defaultdict(list)
    for record in records:
        if float(record["score"]) >= threshold:
            grouped[int(record["image_id"])].append(record)

    output = []
    for image_id, image_records in grouped.items():
        image_records.sort(
            key=lambda record: float(record["score"]),
            reverse=True,
        )
        output.extend(image_records[:FINAL_MAX_DETECTIONS])
    return output

def pair_records_for_keyframe(cache_entry, image_id):
    records = []
    width = int(cache_entry["width"])
    height = int(cache_entry["height"])

    for box, score, label, query_id in zip(
        cache_entry["pair_boxes"],
        cache_entry["pair_scores"],
        cache_entry["pair_labels"],
        cache_entry["pair_query_ids"],
    ):
        clipped = clip_xyxy(box, width, height)
        if clipped is None:
            continue
        x1, y1, x2, y2 = clipped
        records.append({
            "image_id": int(image_id),
            "category_id": int(label) + 1,
            "bbox": [
                round(x1, 2),
                round(y1, 2),
                round(x2 - x1, 2),
                round(y2 - y1, 2),
            ],
            "score": float(score),
            "_query_id": int(query_id),
        })
    return records

def strip_internal_fields(records):
    cleaned = []
    for record in records:
        cleaned.append({
            "image_id": int(record["image_id"]),
            "category_id": int(record["category_id"]),
            "bbox": [round(float(value), 2) for value in record["bbox"]],
            "score": round(float(record["score"]), 6),
        })
    return cleaned

baseline_val_internal = []
for keyframe_id in sorted(valid_windows):
    baseline_val_internal.extend(
        pair_records_for_keyframe(
            valid_cache[keyframe_id],
            keyframe_id,
        )
    )

baseline_val_internal = cap_records(baseline_val_internal)
baseline_val_records = strip_internal_fields(baseline_val_internal)

BASELINE_VAL_PATH = Path("/content/temporal_baseline_val.json")
BASELINE_VAL_PATH.write_text(json.dumps(baseline_val_records))

print("Baseline validation detections:", len(baseline_val_records))
print("Baseline validation file:", BASELINE_VAL_PATH)

Baseline validation detections: 17637
Baseline validation file: /content/temporal_baseline_val.json


In [13]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

with redirect_stdout(StringIO()):
    VALID_COCO_GT = COCO(str(VALID_GT_PATH))

def evaluate_coco_predictions(predictions, image_ids=None):
    with redirect_stdout(StringIO()):
        coco_dt = VALID_COCO_GT.loadRes(predictions if predictions else [])
        evaluator = COCOeval(VALID_COCO_GT, coco_dt, "bbox")
        evaluator.params.maxDets = [1, 10, 100]
        if image_ids is not None:
            evaluator.params.imgIds = sorted(map(int, image_ids))
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()

    precision = evaluator.eval["precision"]
    per_class = {}
    for class_index, class_name in enumerate(CLASS_NAMES):
        class_precision = precision[:, :, class_index, 0, -1]
        valid_values = class_precision[class_precision > -1]
        per_class[f"AP_{class_name}"] = (
            float(valid_values.mean())
            if valid_values.size
            else float("nan")
        )

    return {
        "coco_mAP50_95": float(evaluator.stats[0]),
        "coco_mAP50": float(evaluator.stats[1]),
        "coco_mAP75": float(evaluator.stats[2]),
        "coco_s_map": float(evaluator.stats[3]),
        "coco_m_map": float(evaluator.stats[4]),
        "coco_l_map": float(evaluator.stats[5]),
        "coco_mAR100": float(evaluator.stats[8]),
        **per_class,
    }

def xywh_to_xyxy_array(boxes):
    boxes = np.asarray(boxes, dtype=np.float32)
    output = boxes.copy()
    output[:, 2] = boxes[:, 0] + boxes[:, 2]
    output[:, 3] = boxes[:, 1] + boxes[:, 3]
    return output

def box_iou_matrix(boxes_a, boxes_b):
    boxes_a = np.asarray(boxes_a, dtype=np.float32)
    boxes_b = np.asarray(boxes_b, dtype=np.float32)
    if len(boxes_a) == 0 or len(boxes_b) == 0:
        return np.zeros((len(boxes_a), len(boxes_b)), dtype=np.float32)

    top_left = np.maximum(boxes_a[:, None, :2], boxes_b[None, :, :2])
    bottom_right = np.minimum(boxes_a[:, None, 2:], boxes_b[None, :, 2:])
    intersection_wh = np.maximum(bottom_right - top_left, 0.0)
    intersection = intersection_wh[..., 0] * intersection_wh[..., 1]

    area_a = np.maximum(boxes_a[:, 2] - boxes_a[:, 0], 0.0) * np.maximum(
        boxes_a[:, 3] - boxes_a[:, 1],
        0.0,
    )
    area_b = np.maximum(boxes_b[:, 2] - boxes_b[:, 0], 0.0) * np.maximum(
        boxes_b[:, 3] - boxes_b[:, 1],
        0.0,
    )
    union = area_a[:, None] + area_b[None, :] - intersection
    return intersection / np.maximum(union, EPS)

def interpolated_ap_101(recalls, precisions):
    recall_points = np.linspace(0.0, 1.0, 101)
    values = []
    for recall_point in recall_points:
        mask = recalls >= recall_point
        values.append(float(precisions[mask].max()) if mask.any() else 0.0)
    return float(np.mean(values))

def evaluate_hungarian_map(predictions):
    gt_by_image_class = defaultdict(list)
    for annotation in valid_gt["annotations"]:
        gt_by_image_class[
            (int(annotation["image_id"]), int(annotation["category_id"]))
        ].append(annotation["bbox"])

    pred_by_image_class = defaultdict(list)
    for prediction_index, prediction in enumerate(predictions):
        pred_by_image_class[
            (int(prediction["image_id"]), int(prediction["category_id"]))
        ].append((prediction_index, prediction))

    thresholds = np.arange(0.50, 0.96, 0.05)
    class_aps = defaultdict(list)

    for category_id in range(1, NUM_CLASSES + 1):
        category_predictions = [
            (index, prediction)
            for index, prediction in enumerate(predictions)
            if int(prediction["category_id"]) == category_id
        ]
        num_gt = sum(
            len(boxes)
            for (image_id, cat), boxes in gt_by_image_class.items()
            if cat == category_id
        )
        if num_gt == 0:
            continue

        for threshold in thresholds:
            tp_flags = np.zeros(len(predictions), dtype=np.uint8)

            image_ids = {
                image_id
                for image_id, cat in set(gt_by_image_class) | set(pred_by_image_class)
                if cat == category_id
            }

            for image_id in image_ids:
                gt_boxes_xywh = gt_by_image_class.get(
                    (image_id, category_id),
                    [],
                )
                image_predictions = pred_by_image_class.get(
                    (image_id, category_id),
                    [],
                )

                if not gt_boxes_xywh or not image_predictions:
                    continue

                gt_boxes = xywh_to_xyxy_array(gt_boxes_xywh)
                pred_boxes = xywh_to_xyxy_array([
                    prediction["bbox"]
                    for _, prediction in image_predictions
                ])
                ious = box_iou_matrix(pred_boxes, gt_boxes)
                rows, columns = linear_sum_assignment(-ious)

                for row, column in zip(rows, columns):
                    if ious[row, column] >= threshold:
                        global_prediction_index = image_predictions[row][0]
                        tp_flags[global_prediction_index] = 1

            ranked = sorted(
                category_predictions,
                key=lambda item: float(item[1]["score"]),
                reverse=True,
            )
            if not ranked:
                class_aps[category_id].append(0.0)
                continue

            ranked_indices = [index for index, _ in ranked]
            tp = tp_flags[ranked_indices].astype(np.float32)
            fp = 1.0 - tp
            cumulative_tp = np.cumsum(tp)
            cumulative_fp = np.cumsum(fp)
            recalls = cumulative_tp / max(num_gt, 1)
            precisions = cumulative_tp / np.maximum(
                cumulative_tp + cumulative_fp,
                EPS,
            )
            class_aps[category_id].append(
                interpolated_ap_101(recalls, precisions)
            )

    per_class = {
        f"hungarian_AP_{CLASS_NAMES[category_id - 1]}": float(np.mean(values))
        for category_id, values in class_aps.items()
    }
    overall = (
        float(np.mean(list(per_class.values())))
        if per_class
        else float("nan")
    )
    return {
        "hungarian_mAP50_95": overall,
        **per_class,
    }

baseline_coco_metrics = evaluate_coco_predictions(baseline_val_records)
baseline_hungarian_metrics = evaluate_hungarian_map(baseline_val_records)

print("Baseline COCO metrics:")
print(baseline_coco_metrics)
print("Baseline challenge-style Hungarian metrics:")
print(baseline_hungarian_metrics)

assert (
    abs(
        baseline_coco_metrics["coco_mAP50_95"]
        - EXPECTED_BASELINE_MAP
    )
    <= BASELINE_MAP_TOLERANCE
), (
    "Baseline parity failed. Do not continue to temporal scoring until "
    "the raw/public inference and class mapping are inspected."
)

Baseline COCO metrics:
{'coco_mAP50_95': 0.8865734839610304, 'coco_mAP50': 0.9923511765965171, 'coco_mAP75': 0.9602107819658905, 'coco_s_map': 0.7088010879428958, 'coco_m_map': 0.8789460932484524, 'coco_l_map': 0.9700340219867589, 'coco_mAR100': 0.9039587324843156, 'AP_bee': 0.8540005444799628, 'AP_bumblebee': 0.9613701750609843, 'AP_hoverfly': 0.7812931263966444, 'AP_moth': 0.9496300899065296}
Baseline challenge-style Hungarian metrics:
{'hungarian_mAP50_95': 0.8340456495296653, 'hungarian_AP_bee': 0.6611565831126553, 'hungarian_AP_bumblebee': 0.9613371711556274, 'hungarian_AP_hoverfly': 0.7640587540527825, 'hungarian_AP_moth': 0.949630089797596}


## 11. Motion-aware class-agnostic association

Each current-frame query is an anchor. Hungarian matching proceeds backward through the five previous frames. The cost uses normalized center displacement, size, aspect ratio, IoU, optional constant-velocity consistency, and only a small class-disagreement penalty.

In [14]:
def box_centers(boxes):
    boxes = np.asarray(boxes, dtype=np.float32)
    return (boxes[:, :2] + boxes[:, 2:]) / 2.0

def box_areas(boxes):
    boxes = np.asarray(boxes, dtype=np.float32)
    return np.maximum(boxes[:, 2] - boxes[:, 0], 1.0) * np.maximum(
        boxes[:, 3] - boxes[:, 1],
        1.0,
    )

def box_aspects(boxes):
    boxes = np.asarray(boxes, dtype=np.float32)
    widths = np.maximum(boxes[:, 2] - boxes[:, 0], 1.0)
    heights = np.maximum(boxes[:, 3] - boxes[:, 1], 1.0)
    return widths / heights

def predicted_track_centers(track_histories, target_time):
    predictions = []
    for history in track_histories:
        # History rows: (time, box, label).
        ordered = sorted(history, key=lambda item: item[0], reverse=True)
        last_time, last_box, _ = ordered[-1]
        last_center = box_centers(np.asarray([last_box]))[0]

        if len(ordered) >= 2:
            previous_time, previous_box, _ = ordered[-2]
            previous_center = box_centers(np.asarray([previous_box]))[0]
            time_delta = last_time - previous_time
            if time_delta != 0:
                velocity = (last_center - previous_center) / time_delta
                prediction = last_center + velocity * (target_time - last_time)
            else:
                prediction = last_center
        else:
            prediction = last_center

        predictions.append(prediction)

    return np.asarray(predictions, dtype=np.float32)

def association_cost_matrix(track_histories, candidate_entry, target_time):
    candidate_boxes = np.asarray(
        candidate_entry["proposal_boxes"],
        dtype=np.float32,
    )
    candidate_labels = np.asarray(
        candidate_entry["proposal_labels"],
        dtype=np.int16,
    )

    if not track_histories or len(candidate_boxes) == 0:
        return np.zeros(
            (len(track_histories), len(candidate_boxes)),
            dtype=np.float32,
        )

    # We are moving backward in time. The reference for the next match is
    # the oldest/most-negative time already matched, not the t=0 anchor.
    last_boxes = np.asarray([
        min(history, key=lambda item: item[0])[1]
        for history in track_histories
    ], dtype=np.float32)
    last_labels = np.asarray([
        min(history, key=lambda item: item[0])[2]
        for history in track_histories
    ], dtype=np.int16)

    last_centers = box_centers(last_boxes)
    candidate_centers = box_centers(candidate_boxes)
    predicted_centers = predicted_track_centers(
        track_histories,
        target_time,
    )

    last_areas = box_areas(last_boxes)
    candidate_areas = box_areas(candidate_boxes)
    scale = np.maximum(
        np.sqrt(last_areas)[:, None],
        np.sqrt(candidate_areas)[None, :],
    )
    scale = np.maximum(scale, ASSOC["min_scale_px"])

    center_norm = np.linalg.norm(
        last_centers[:, None, :] - candidate_centers[None, :, :],
        axis=2,
    ) / scale

    motion_norm = np.linalg.norm(
        predicted_centers[:, None, :] - candidate_centers[None, :, :],
        axis=2,
    ) / scale

    log_area = np.abs(
        np.log(
            np.maximum(last_areas[:, None], EPS)
            / np.maximum(candidate_areas[None, :], EPS)
        )
    )

    last_aspects = box_aspects(last_boxes)
    candidate_aspects = box_aspects(candidate_boxes)
    log_aspect = np.abs(
        np.log(
            np.maximum(last_aspects[:, None], EPS)
            / np.maximum(candidate_aspects[None, :], EPS)
        )
    )

    iou = box_iou_matrix(last_boxes, candidate_boxes)
    class_mismatch = (
        last_labels[:, None] != candidate_labels[None, :]
    ).astype(np.float32)

    has_velocity = np.asarray([
        len(history) >= 2
        for history in track_histories
    ], dtype=np.float32)[:, None]

    cost = (
        ASSOC["w_center"] * center_norm
        + ASSOC["w_area"] * log_area
        + ASSOC["w_aspect"] * log_aspect
        + ASSOC["w_iou"] * (1.0 - iou)
        + ASSOC["w_motion"] * motion_norm * has_velocity
        + ASSOC["class_mismatch_penalty"] * class_mismatch
    )

    invalid = (
        (center_norm > ASSOC["max_center_norm"])
        | (log_area > ASSOC["max_log_area"])
        | (log_aspect > ASSOC["max_log_aspect"])
        | (
            (has_velocity > 0)
            & (motion_norm > ASSOC["max_motion_norm"])
        )
    )
    cost[invalid] = 1e6
    return cost.astype(np.float32)

def associate_anchor_frame(anchor_entry, prior_entries):
    anchor_boxes = np.asarray(
        anchor_entry["proposal_boxes"],
        dtype=np.float32,
    )
    anchor_labels = np.asarray(
        anchor_entry["proposal_labels"],
        dtype=np.int16,
    )
    anchor_query_ids = np.asarray(
        anchor_entry["proposal_query_ids"],
        dtype=np.int32,
    )

    num_anchors = len(anchor_boxes)
    histories = [
        [(0, anchor_boxes[index], int(anchor_labels[index]))]
        for index in range(num_anchors)
    ]
    gaps = np.zeros(num_anchors, dtype=np.int16)
    match_indices = np.full((num_anchors, len(prior_entries)), -1, dtype=np.int32)
    match_costs = np.full((num_anchors, len(prior_entries)), np.inf, dtype=np.float32)

    for lag, candidate_entry in enumerate(prior_entries, start=1):
        active_anchor_indices = [
            index
            for index in range(num_anchors)
            if gaps[index] <= ASSOC["max_gap"]
        ]
        if not active_anchor_indices:
            break

        active_histories = [
            histories[index]
            for index in active_anchor_indices
        ]
        cost = association_cost_matrix(
            active_histories,
            candidate_entry,
            target_time=-lag,
        )

        matched_active_rows = set()

        if cost.size:
            rows, columns = linear_sum_assignment(cost)
            for row, column in zip(rows, columns):
                value = float(cost[row, column])
                if value >= ASSOC["max_cost"] or value >= 1e5:
                    continue

                anchor_index = active_anchor_indices[row]
                candidate_box = candidate_entry["proposal_boxes"][column]
                candidate_label = int(
                    candidate_entry["proposal_labels"][column]
                )

                histories[anchor_index].append(
                    (-lag, candidate_box, candidate_label)
                )
                match_indices[anchor_index, lag - 1] = int(column)
                match_costs[anchor_index, lag - 1] = value
                gaps[anchor_index] = 0
                matched_active_rows.add(row)

        for row, anchor_index in enumerate(active_anchor_indices):
            if row not in matched_active_rows:
                gaps[anchor_index] += 1

    reliabilities = np.exp(
        -np.where(np.isfinite(match_costs), match_costs, 100.0)
    ).astype(np.float32)
    reliabilities[match_indices < 0] = 0.0

    weights = np.asarray(
        [RHO ** lag for lag in range(len(prior_entries))],
        dtype=np.float32,
    )

    t_hard = np.zeros((num_anchors, NUM_CLASSES), dtype=np.float32)
    t_soft = np.zeros((num_anchors, NUM_CLASSES), dtype=np.float32)
    object_terms = np.ones(num_anchors, dtype=np.float32)

    match_labels = np.full_like(match_indices, -1, dtype=np.int16)
    match_scores = np.zeros_like(match_costs, dtype=np.float32)

    for lag_index, candidate_entry in enumerate(prior_entries):
        valid_anchor_indices = np.where(match_indices[:, lag_index] >= 0)[0]
        for anchor_index in valid_anchor_indices:
            candidate_index = match_indices[anchor_index, lag_index]
            reliability = reliabilities[anchor_index, lag_index]
            time_weight = weights[lag_index]
            score = float(candidate_entry["proposal_scores"][candidate_index])
            label = int(candidate_entry["proposal_labels"][candidate_index])
            class_scores = np.asarray(
                candidate_entry["proposal_class_scores"][candidate_index],
                dtype=np.float32,
            )

            match_labels[anchor_index, lag_index] = label
            match_scores[anchor_index, lag_index] = score

            hard_value = np.clip(
                time_weight * reliability * score,
                0.0,
                1.0,
            )
            t_hard[anchor_index, label] = (
                1.0
                - (1.0 - t_hard[anchor_index, label])
                * (1.0 - hard_value)
            )

            for class_index in range(NUM_CLASSES):
                soft_value = np.clip(
                    time_weight
                    * reliability
                    * float(class_scores[class_index]),
                    0.0,
                    1.0,
                )
                t_soft[anchor_index, class_index] = (
                    1.0
                    - (1.0 - t_soft[anchor_index, class_index])
                    * (1.0 - soft_value)
                )

            object_terms[anchor_index] *= (1.0 - hard_value)

    t_object = 1.0 - object_terms
    continuity = (
        (
            reliabilities
            * (match_indices >= 0)
            * weights[None, :]
        ).sum(axis=1)
        / max(float(weights.sum()), EPS)
    ).astype(np.float32)
    track_reliability = (
        t_object * (0.5 + 0.5 * continuity)
    ).astype(np.float32)

    weighted_denominator = (
        reliabilities * weights[None, :]
    ).sum(axis=1)
    historical_weighted_mean = (
        match_scores * reliabilities * weights[None, :]
    ).sum(axis=1) / np.maximum(weighted_denominator, EPS)
    historical_max = match_scores.max(axis=1) if match_scores.size else np.zeros(num_anchors)

    return {
        "query_ids": anchor_query_ids,
        "match_indices": match_indices,
        "match_costs": match_costs,
        "match_reliabilities": reliabilities,
        "match_labels": match_labels,
        "match_scores": match_scores,
        "T_hard": t_hard,
        "T_soft": t_soft,
        "T_object": t_object.astype(np.float32),
        "continuity": continuity,
        "P": track_reliability,
        "n_matches": (match_indices >= 0).sum(axis=1).astype(np.int16),
        "recent_match": (
            (match_indices[:, :2] >= 0)
            & (reliabilities[:, :2] >= MIN_RELIABLE_MATCH)
        ).any(axis=1),
        "historical_weighted_mean": historical_weighted_mean.astype(np.float32),
        "historical_max": historical_max.astype(np.float32),
    }

def associate_keyframe(window, cache):
    frame_ids = window["frame_ids"]
    current_id = int(frame_ids[-1])
    prior_ids_recent_first = [
        int(image_id)
        for image_id in reversed(frame_ids[:-1])
    ]
    return associate_anchor_frame(
        cache[current_id],
        [cache[image_id] for image_id in prior_ids_recent_first],
    )

## 12. Resumable track-feature cache

In [15]:
def track_shard_paths(split_name):
    directory = TRACK_DIR / split_name
    directory.mkdir(parents=True, exist_ok=True)
    return sorted(directory.glob("tracks_*.pkl.gz"))

def load_track_cache(split_name):
    merged = {}
    for shard_path in tqdm(
        track_shard_paths(split_name),
        desc=f"load {split_name} track shards",
    ):
        shard = gzip_pickle_load(shard_path)
        merged.update({int(key): value for key, value in shard.items()})
    return merged

def run_track_cache(split_name, windows, cache):
    output_dir = TRACK_DIR / split_name
    output_dir.mkdir(parents=True, exist_ok=True)

    existing = load_track_cache(split_name)
    processed_ids = set(existing)
    del existing
    gc.collect()

    pending_ids = [
        keyframe_id
        for keyframe_id in sorted(windows)
        if keyframe_id not in processed_ids
    ]

    print(
        f"{split_name}: {len(processed_ids)} tracklets cached, "
        f"{len(pending_ids)} pending."
    )

    next_shard_index = len(track_shard_paths(split_name))
    shard = {}

    for keyframe_id in tqdm(
        pending_ids,
        desc=f"{split_name} motion-aware tracklets",
    ):
        shard[keyframe_id] = associate_keyframe(
            windows[keyframe_id],
            cache,
        )

        if len(shard) >= TRACK_SHARD_SIZE:
            destination = (
                output_dir / f"tracks_{next_shard_index:05d}.pkl.gz"
            )
            atomic_gzip_pickle_dump(shard, destination)
            next_shard_index += 1
            shard = {}
            gc.collect()

    if shard:
        destination = output_dir / f"tracks_{next_shard_index:05d}.pkl.gz"
        atomic_gzip_pickle_dump(shard, destination)

    final_tracks = load_track_cache(split_name)
    assert set(final_tracks) == set(windows)
    print(f"{split_name} track cache complete:", len(final_tracks))
    return final_tracks

if RUN_VALID_TRACK_CACHE:
    valid_tracks = run_track_cache(
        "valid",
        valid_windows,
        valid_cache,
    )
else:
    valid_tracks = load_track_cache("valid")

if RUN_TEST_TRACK_CACHE:
    test_tracks = run_track_cache(
        "test",
        test_windows,
        test_cache,
    )
else:
    test_tracks = load_track_cache("test")

print("Validation tracklets:", len(valid_tracks))
print("Test tracklets:", len(test_tracks))

load valid track shards: 0it [00:00, ?it/s]

valid: 0 tracklets cached, 932 pending.


valid motion-aware tracklets:   0%|          | 0/932 [00:00<?, ?it/s]

load valid track shards:   0%|          | 0/7 [00:00<?, ?it/s]

valid track cache complete: 932


load test track shards: 0it [00:00, ?it/s]

test: 0 tracklets cached, 4763 pending.


test motion-aware tracklets:   0%|          | 0/4763 [00:00<?, ?it/s]

load test track shards:   0%|          | 0/32 [00:00<?, ?it/s]

test track cache complete: 4763
Validation tracklets: 932
Test tracklets: 4763


## 13. Hoverfly rescueability diagnostic

This measures whether failed current-frame hoverflies contain usable hoverfly evidence in their five-frame histories. It separates:

- correctly detected hoverflies
- localized but misclassified hoverflies
- low-score current proposals
- true localization misses

In [16]:
def annotations_by_image(coco):
    output = defaultdict(list)
    for annotation in coco["annotations"]:
        output[int(annotation["image_id"])].append(annotation)
    return output

VALID_ANNOTATIONS_BY_IMAGE = annotations_by_image(valid_gt)

def query_id_to_proposal_index(cache_entry):
    return {
        int(query_id): index
        for index, query_id in enumerate(
            cache_entry["proposal_query_ids"]
        )
    }

def hoverfly_rescueability_diagnostic():
    rows = []
    hoverfly_category_id = CATEGORY_IDS["hoverfly"]

    for keyframe_id in sorted(valid_windows):
        cache_entry = valid_cache[keyframe_id]
        track_entry = valid_tracks[keyframe_id]
        proposal_boxes = np.asarray(cache_entry["proposal_boxes"])
        proposal_scores = np.asarray(cache_entry["proposal_scores"])
        proposal_labels = np.asarray(cache_entry["proposal_labels"])

        gt_hoverflies = [
            annotation
            for annotation in VALID_ANNOTATIONS_BY_IMAGE[keyframe_id]
            if int(annotation["category_id"]) == hoverfly_category_id
        ]

        for annotation in gt_hoverflies:
            gt_box = xywh_to_xyxy_array([annotation["bbox"]])
            if len(proposal_boxes):
                ious = box_iou_matrix(proposal_boxes, gt_box)[:, 0]
                best_index = int(np.argmax(ious))
                best_iou = float(ious[best_index])
            else:
                best_index = -1
                best_iou = 0.0

            if best_index >= 0:
                current_label = int(proposal_labels[best_index])
                current_score = float(proposal_scores[best_index])
                matched_labels = track_entry["match_labels"][best_index]
                matched_reliabilities = track_entry["match_reliabilities"][best_index]

                reliable = matched_reliabilities >= MIN_RELIABLE_MATCH
                prior_hoverfly = (
                    (matched_labels == hoverfly_category_id - 1)
                    & reliable
                )
                prior_bee = (
                    (matched_labels == CATEGORY_IDS["bee"] - 1)
                    & reliable
                )
                prior_hoverfly_count = int(prior_hoverfly.sum())
                recent_hoverfly = bool(prior_hoverfly[:2].any())
                all_reliable_bee = bool(
                    reliable.any()
                    and np.all(
                        matched_labels[reliable]
                        == CATEGORY_IDS["bee"] - 1
                    )
                )
            else:
                current_label = -1
                current_score = 0.0
                prior_hoverfly_count = 0
                recent_hoverfly = False
                all_reliable_bee = False

            correctly_detected = (
                best_iou >= 0.50
                and current_label == hoverfly_category_id - 1
                and current_score >= BASELINE_SUBMISSION_THRESHOLD
            )
            localized_wrong_class = (
                best_iou >= 0.50
                and current_label != hoverfly_category_id - 1
            )
            low_score_correct_proposal = (
                best_iou >= 0.50
                and current_label == hoverfly_category_id - 1
                and current_score < BASELINE_SUBMISSION_THRESHOLD
            )
            localization_miss = best_iou < 0.50

            rescueable_class = (
                localized_wrong_class
                and prior_hoverfly_count >= MIN_CLASS_MATCHES
                and recent_hoverfly
            )
            rescueable_score = (
                low_score_correct_proposal
                and prior_hoverfly_count >= 1
            )

            rows.append({
                "image_id": keyframe_id,
                "annotation_id": int(annotation["id"]),
                "best_iou": best_iou,
                "current_class": (
                    CLASS_NAMES[current_label]
                    if 0 <= current_label < NUM_CLASSES
                    else "missing"
                ),
                "current_score": current_score,
                "correctly_detected": correctly_detected,
                "localized_wrong_class": localized_wrong_class,
                "low_score_correct_proposal": low_score_correct_proposal,
                "localization_miss": localization_miss,
                "prior_hoverfly_count": prior_hoverfly_count,
                "recent_hoverfly": recent_hoverfly,
                "all_reliable_history_bee": all_reliable_bee,
                "rescueable_class": rescueable_class,
                "rescueable_score": rescueable_score,
            })

    dataframe = pd.DataFrame(rows)
    failed = dataframe[~dataframe["correctly_detected"]]
    rescueable = failed[
        dataframe.loc[failed.index, "rescueable_class"]
        | dataframe.loc[failed.index, "rescueable_score"]
    ]

    rescueable_fraction = (
        len(rescueable) / len(failed)
        if len(failed)
        else 0.0
    )

    summary = {
        "total_hoverflies": int(len(dataframe)),
        "correctly_detected": int(dataframe["correctly_detected"].sum()),
        "localized_wrong_class": int(dataframe["localized_wrong_class"].sum()),
        "low_score_correct_proposal": int(dataframe["low_score_correct_proposal"].sum()),
        "localization_miss": int(dataframe["localization_miss"].sum()),
        "rescueable_class": int(dataframe["rescueable_class"].sum()),
        "rescueable_score": int(dataframe["rescueable_score"].sum()),
        "failed_total": int(len(failed)),
        "rescueable_failed": int(len(rescueable)),
        "rescueable_fraction": float(rescueable_fraction),
        "all_reliable_history_bee": int(dataframe["all_reliable_history_bee"].sum()),
    }

    return dataframe, summary

if RUN_HOVERFLY_DIAGNOSTIC:
    hoverfly_df, hoverfly_summary = hoverfly_rescueability_diagnostic()
    hoverfly_df.to_csv(
        REPORT_DIR / "hoverfly_rescueability_details.csv",
        index=False,
    )
    (REPORT_DIR / "hoverfly_rescueability_summary.json").write_text(
        json.dumps(hoverfly_summary, indent=2)
    )
    print(json.dumps(hoverfly_summary, indent=2))
    display(hoverfly_df.head(20))
else:
    hoverfly_summary = json.loads(
        (REPORT_DIR / "hoverfly_rescueability_summary.json").read_text()
    )

HOVERFLY_CLASS_CORRECTION_SUPPORTED = (
    float(hoverfly_summary["rescueable_fraction"])
    >= MIN_RESCUEABLE_FRACTION
)

print(
    "Hoverfly class correction supported by diagnostic:",
    HOVERFLY_CLASS_CORRECTION_SUPPORTED,
)

{
  "total_hoverflies": 95,
  "correctly_detected": 79,
  "localized_wrong_class": 15,
  "low_score_correct_proposal": 0,
  "localization_miss": 1,
  "rescueable_class": 1,
  "rescueable_score": 0,
  "failed_total": 16,
  "rescueable_failed": 1,
  "rescueable_fraction": 0.0625,
  "all_reliable_history_bee": 19
}


,image_id,annotation_id,best_iou,current_class,current_score,correctly_detected,localized_wrong_class,low_score_correct_proposal,localization_miss,prior_hoverfly_count,recent_hoverfly,all_reliable_history_bee,rescueable_class,rescueable_score
0,270,47,0.864131,hoverfly,0.897676,True,False,False,False,5,True,False,False,False
1,438,76,0.895875,hoverfly,0.780615,True,False,False,False,4,True,False,False,False
2,444,77,0.524652,bee,0.008084,False,True,False,False,0,False,True,False,False
3,450,79,0.955601,hoverfly,0.930636,True,False,False,False,0,False,False,False,False
4,456,80,0.537559,bee,0.007057,False,True,False,False,0,False,True,False,False
5,462,81,0.942942,hoverfly,0.838193,True,False,False,False,5,True,False,False,False
6,468,82,0.699378,bee,0.008321,False,True,False,False,0,False,True,False,False
7,474,83,0.941256,hoverfly,0.919486,True,False,False,False,1,True,False,False,False
8,480,84,0.880299,hoverfly,0.767452,True,False,False,False,2,True,False,False,False
9,486,85,0.957485,hoverfly,0.881427,True,False,False,False,5,True,False,False,False


Hoverfly class correction supported by diagnostic: False


## 14. Fixed temporal scoring and class-fusion candidates

In [17]:
def corrected_rerank_score(score, track_reliability, lambda_value, delta_value):
    score = np.asarray(score, dtype=np.float32)
    track_reliability = np.asarray(track_reliability, dtype=np.float32)
    return np.clip(
        score
        + lambda_value * (1.0 - score) * track_reliability
        - delta_value
        * score
        * np.square(1.0 - score)
        * (1.0 - track_reliability),
        0.0,
        1.0,
    )

def softmax_numpy(log_values):
    shifted = log_values - np.max(log_values)
    values = np.exp(shifted)
    return values / np.maximum(values.sum(), EPS)

def track_row_lookup(track_entry):
    return {
        int(query_id): index
        for index, query_id in enumerate(track_entry["query_ids"])
    }

def pair_candidate_records(
    keyframe_id,
    cache_entry,
    track_entry,
    mode,
    lambda_value=DEFAULT_LAMBDA,
    delta_value=DEFAULT_DELTA,
):
    track_lookup = track_row_lookup(track_entry)
    output = []

    for box, original_score, label, query_id in zip(
        cache_entry["pair_boxes"],
        cache_entry["pair_scores"],
        cache_entry["pair_labels"],
        cache_entry["pair_query_ids"],
    ):
        track_index = track_lookup.get(int(query_id))
        if track_index is None:
            p_track = 0.0
            hist_mean = 0.0
            hist_max = 0.0
        else:
            p_track = float(track_entry["P"][track_index])
            hist_mean = float(
                track_entry["historical_weighted_mean"][track_index]
            )
            hist_max = float(track_entry["historical_max"][track_index])

        original_score = float(original_score)

        if mode == "baseline":
            new_score = original_score
        elif mode == "seq_mean":
            if p_track > 0:
                new_score = (
                    original_score + p_track * hist_mean
                ) / (1.0 + p_track)
            else:
                new_score = original_score
        elif mode == "seq_bounded_max":
            new_score = original_score + BOUNDED_MAX_BLEND * max(
                0.0,
                hist_max - original_score,
            )
        elif mode == "rerank":
            new_score = float(corrected_rerank_score(
                original_score,
                p_track,
                lambda_value,
                delta_value,
            ))
        else:
            raise ValueError(mode)

        clipped = clip_xyxy(
            box,
            int(cache_entry["width"]),
            int(cache_entry["height"]),
        )
        if clipped is None:
            continue
        x1, y1, x2, y2 = clipped

        output.append({
            "image_id": int(keyframe_id),
            "category_id": int(label) + 1,
            "bbox": [
                round(x1, 2),
                round(y1, 2),
                round(x2 - x1, 2),
                round(y2 - y1, 2),
            ],
            "score": float(new_score),
            "_query_id": int(query_id),
        })

    return output

def class_fusion_records(
    keyframe_id,
    cache_entry,
    track_entry,
    lambda_value=DEFAULT_LAMBDA,
    delta_value=DEFAULT_DELTA,
    fallback_rule=False,
):
    output = []

    for proposal_index, (
        query_id,
        box,
        current_score,
        current_label,
        current_class_scores,
    ) in enumerate(zip(
        cache_entry["proposal_query_ids"],
        cache_entry["proposal_boxes"],
        cache_entry["proposal_scores"],
        cache_entry["proposal_labels"],
        cache_entry["proposal_class_scores"],
    )):
        p_track = float(track_entry["P"][proposal_index])
        new_score = float(corrected_rerank_score(
            current_score,
            p_track,
            lambda_value,
            delta_value,
        ))
        new_label = int(current_label)

        n_matches = int(track_entry["n_matches"][proposal_index])
        recent_match = bool(track_entry["recent_match"][proposal_index])
        t_hard = np.asarray(
            track_entry["T_hard"][proposal_index],
            dtype=np.float32,
        )
        t_soft = np.asarray(
            track_entry["T_soft"][proposal_index],
            dtype=np.float32,
        )

        can_consider_relabel = (
            HOVERFLY_CLASS_CORRECTION_SUPPORTED
            and n_matches >= MIN_CLASS_MATCHES
            and recent_match
            and p_track >= MIN_CLASS_TRACK_RELIABILITY
            and current_label
            in {
                CATEGORY_IDS["bee"] - 1,
                CATEGORY_IDS["hoverfly"] - 1,
            }
        )

        if can_consider_relabel:
            bee_index = CATEGORY_IDS["bee"] - 1
            hoverfly_index = CATEGORY_IDS["hoverfly"] - 1

            if (
                cache_entry.get("has_full_class_scores", False)
                and not fallback_rule
            ):
                current_distribution = np.asarray(
                    current_class_scores,
                    dtype=np.float64,
                )
                current_distribution = (
                    current_distribution + EPS
                ) / np.maximum(current_distribution.sum(), EPS)

                temporal_distribution = (
                    t_soft.astype(np.float64) + EPS
                ) / np.maximum(t_soft.sum(), EPS)

                alpha = ALPHA_MAX * p_track
                fused_log = (
                    (1.0 - alpha) * np.log(current_distribution + EPS)
                    + alpha * np.log(temporal_distribution + EPS)
                )
                fused = softmax_numpy(fused_log)

                pair_scores = {
                    bee_index: float(fused[bee_index]),
                    hoverfly_index: float(fused[hoverfly_index]),
                }
                proposed_label = max(pair_scores, key=pair_scores.get)
                support_margin = abs(
                    float(t_soft[hoverfly_index] - t_soft[bee_index])
                )

                if (
                    proposed_label != current_label
                    and support_margin >= MIN_CLASS_SUPPORT_MARGIN
                ):
                    new_label = int(proposed_label)
            else:
                # Winning-label fallback: only bee -> hoverfly, with a strict
                # temporal majority and support-margin requirement.
                hoverfly_support = float(t_hard[hoverfly_index])
                bee_support = float(t_hard[bee_index])
                prior_labels = track_entry["match_labels"][proposal_index]
                prior_reliability = track_entry[
                    "match_reliabilities"
                ][proposal_index]
                reliable_hoverfly_count = int(
                    (
                        (prior_labels == hoverfly_index)
                        & (prior_reliability >= MIN_RELIABLE_MATCH)
                    ).sum()
                )

                if (
                    current_label == bee_index
                    and reliable_hoverfly_count >= MIN_CLASS_MATCHES
                    and hoverfly_support - bee_support
                    >= MIN_CLASS_SUPPORT_MARGIN
                ):
                    new_label = hoverfly_index

        clipped = clip_xyxy(
            box,
            int(cache_entry["width"]),
            int(cache_entry["height"]),
        )
        if clipped is None:
            continue
        x1, y1, x2, y2 = clipped

        output.append({
            "image_id": int(keyframe_id),
            "category_id": int(new_label) + 1,
            "bbox": [
                round(x1, 2),
                round(y1, 2),
                round(x2 - x1, 2),
                round(y2 - y1, 2),
            ],
            "score": float(new_score),
            "_query_id": int(query_id),
        })

    return output

CANDIDATE_CONFIGS = {
    "baseline": {"type": "pair", "mode": "baseline"},
    "seq_mean": {"type": "pair", "mode": "seq_mean"},
    "seq_bounded_max": {
        "type": "pair",
        "mode": "seq_bounded_max",
    },
}

for lambda_value in LAMBDA_VALUES:
    for delta_value in DELTA_VALUES:
        name = (
            f"rerank_l{int(lambda_value * 100):02d}"
            f"_d{int(delta_value * 100):02d}"
        )
        CANDIDATE_CONFIGS[name] = {
            "type": "pair",
            "mode": "rerank",
            "lambda": lambda_value,
            "delta": delta_value,
        }

if HOVERFLY_CLASS_CORRECTION_SUPPORTED:
    if RAW_INFERENCE_OK:
        CANDIDATE_CONFIGS["classfusion_l35_d15"] = {
            "type": "classfusion",
            "lambda": DEFAULT_LAMBDA,
            "delta": DEFAULT_DELTA,
            "fallback_rule": False,
        }

    CANDIDATE_CONFIGS["hoverfly_rule_l35_d15"] = {
        "type": "classfusion",
        "lambda": DEFAULT_LAMBDA,
        "delta": DEFAULT_DELTA,
        "fallback_rule": True,
    }
else:
    print(
        "Hoverfly diagnostic did not meet the rescueability gate; "
        "class-correction candidates are disabled."
    )

print("Candidate configurations:")
for name, config in CANDIDATE_CONFIGS.items():
    print(" ", name, config)

Hoverfly diagnostic did not meet the rescueability gate; class-correction candidates are disabled.
Candidate configurations:
  baseline {'type': 'pair', 'mode': 'baseline'}
  seq_mean {'type': 'pair', 'mode': 'seq_mean'}
  seq_bounded_max {'type': 'pair', 'mode': 'seq_bounded_max'}
  rerank_l25_d00 {'type': 'pair', 'mode': 'rerank', 'lambda': 0.25, 'delta': 0.0}
  rerank_l25_d10 {'type': 'pair', 'mode': 'rerank', 'lambda': 0.25, 'delta': 0.1}
  rerank_l25_d15 {'type': 'pair', 'mode': 'rerank', 'lambda': 0.25, 'delta': 0.15}
  rerank_l35_d00 {'type': 'pair', 'mode': 'rerank', 'lambda': 0.35, 'delta': 0.0}
  rerank_l35_d10 {'type': 'pair', 'mode': 'rerank', 'lambda': 0.35, 'delta': 0.1}
  rerank_l35_d15 {'type': 'pair', 'mode': 'rerank', 'lambda': 0.35, 'delta': 0.15}
  rerank_l45_d00 {'type': 'pair', 'mode': 'rerank', 'lambda': 0.45, 'delta': 0.0}
  rerank_l45_d10 {'type': 'pair', 'mode': 'rerank', 'lambda': 0.45, 'delta': 0.1}
  rerank_l45_d15 {'type': 'pair', 'mode': 'rerank', 'lambda

## 15. Optional aggressive true-box propagation

This only considers a strong track that reaches \(t-1\), continues through at least two older frames, and has no plausible current-frame proposal nearby. Propagated boxes are never mixed into the main reranking candidates; they remain a separate ablation.

In [18]:
def center_size_to_xyxy(center, width, height):
    cx, cy = map(float, center)
    return np.asarray([
        cx - width / 2.0,
        cy - height / 2.0,
        cx + width / 2.0,
        cy + height / 2.0,
    ], dtype=np.float32)

def generate_propagated_records(keyframe_id, window, cache, current_tracks):
    if not ENABLE_TRUE_PROPAGATION:
        return []

    frame_ids = window["frame_ids"]
    current_entry = cache[int(frame_ids[-1])]
    t_minus_1_entry = cache[int(frame_ids[-2])]
    older_entries = [
        cache[int(image_id)]
        for image_id in reversed(frame_ids[:-2])
    ]

    # Previous-frame proposals already claimed by current anchors at lag 1.
    claimed_previous_indices = set(
        int(index)
        for index in current_tracks["match_indices"][:, 0]
        if int(index) >= 0
    )

    previous_tracks = associate_anchor_frame(
        t_minus_1_entry,
        older_entries,
    )

    current_boxes = np.asarray(
        current_entry["proposal_boxes"],
        dtype=np.float32,
    )
    current_centers = box_centers(current_boxes)
    current_areas = box_areas(current_boxes)

    propagated = []

    for previous_index, (
        anchor_box,
        anchor_score,
        anchor_label,
    ) in enumerate(zip(
        t_minus_1_entry["proposal_boxes"],
        t_minus_1_entry["proposal_scores"],
        t_minus_1_entry["proposal_labels"],
    )):
        if previous_index in claimed_previous_indices:
            continue
        if float(anchor_score) < PROP_MIN_ANCHOR_SCORE:
            continue
        if int(previous_tracks["n_matches"][previous_index]) < PROP_MIN_MATCHES:
            continue

        p_track = float(previous_tracks["P"][previous_index])
        if p_track < PROP_MIN_TRACK_RELIABILITY:
            continue

        t_minus_2_index = int(
            previous_tracks["match_indices"][previous_index, 0]
        )
        if t_minus_2_index < 0:
            continue

        box_t1 = np.asarray(anchor_box, dtype=np.float32)
        box_t2 = np.asarray(
            older_entries[0]["proposal_boxes"][t_minus_2_index],
            dtype=np.float32,
        )
        center_t1 = box_centers(np.asarray([box_t1]))[0]
        center_t2 = box_centers(np.asarray([box_t2]))[0]
        predicted_center = center_t1 + (center_t1 - center_t2)

        widths = [
            max(float(box_t1[2] - box_t1[0]), 1.0),
            max(float(box_t2[2] - box_t2[0]), 1.0),
        ]
        heights = [
            max(float(box_t1[3] - box_t1[1]), 1.0),
            max(float(box_t2[3] - box_t2[1]), 1.0),
        ]

        t_minus_3_index = int(
            previous_tracks["match_indices"][previous_index, 1]
        )
        if t_minus_3_index >= 0:
            box_t3 = np.asarray(
                older_entries[1]["proposal_boxes"][t_minus_3_index],
                dtype=np.float32,
            )
            widths.append(max(float(box_t3[2] - box_t3[0]), 1.0))
            heights.append(max(float(box_t3[3] - box_t3[1]), 1.0))

        predicted_box = center_size_to_xyxy(
            predicted_center,
            float(np.median(widths)),
            float(np.median(heights)),
        )

        clipped = clip_xyxy(
            predicted_box,
            int(current_entry["width"]),
            int(current_entry["height"]),
        )
        if clipped is None:
            continue
        predicted_box = np.asarray(clipped, dtype=np.float32)

        # Reject if a current-frame proposal already plausibly represents it.
        if len(current_boxes):
            ious = box_iou_matrix(
                np.asarray([predicted_box]),
                current_boxes,
            )[0]
            predicted_area = box_areas(
                np.asarray([predicted_box])
            )[0]
            scale = np.maximum(
                np.sqrt(np.maximum(predicted_area, 1.0)),
                np.sqrt(np.maximum(current_areas, 1.0)),
            )
            center_norm = np.linalg.norm(
                current_centers - predicted_center[None, :],
                axis=1,
            ) / np.maximum(scale, ASSOC["min_scale_px"])

            if (
                float(ious.max(initial=0.0)) >= PROP_DUP_IOU
                or float(center_norm.min(initial=np.inf))
                <= PROP_DUP_CENTER_NORM
            ):
                continue

        reranked_anchor_score = float(corrected_rerank_score(
            float(anchor_score),
            p_track,
            DEFAULT_LAMBDA,
            DEFAULT_DELTA,
        ))
        propagated_score = (
            PROP_SCORE_DECAY
            * reranked_anchor_score
            * (0.5 + 0.5 * p_track)
        )

        x1, y1, x2, y2 = predicted_box.tolist()
        propagated.append({
            "image_id": int(keyframe_id),
            "category_id": int(anchor_label) + 1,
            "bbox": [
                round(x1, 2),
                round(y1, 2),
                round(x2 - x1, 2),
                round(y2 - y1, 2),
            ],
            "score": float(np.clip(propagated_score, 0.0, 1.0)),
            "_query_id": -1,
            "_propagated": True,
        })

    return propagated

## 16. Generate and evaluate all validation candidates

In [19]:
def generate_candidate_records(
    candidate_name,
    windows,
    cache,
    tracks,
    include_propagation=False,
):
    config = CANDIDATE_CONFIGS[candidate_name]
    records = []

    for keyframe_id in sorted(windows):
        cache_entry = cache[keyframe_id]
        track_entry = tracks[keyframe_id]

        if config["type"] == "pair":
            keyframe_records = pair_candidate_records(
                keyframe_id,
                cache_entry,
                track_entry,
                mode=config["mode"],
                lambda_value=config.get("lambda", DEFAULT_LAMBDA),
                delta_value=config.get("delta", DEFAULT_DELTA),
            )
        elif config["type"] == "classfusion":
            keyframe_records = class_fusion_records(
                keyframe_id,
                cache_entry,
                track_entry,
                lambda_value=config["lambda"],
                delta_value=config["delta"],
                fallback_rule=config["fallback_rule"],
            )
        else:
            raise ValueError(config)

        if include_propagation:
            keyframe_records.extend(
                generate_propagated_records(
                    keyframe_id,
                    windows[keyframe_id],
                    cache,
                    track_entry,
                )
            )

        records.extend(keyframe_records)

    return cap_records(records)

def group_split_ids(windows):
    groups = sorted({
        window["group"]
        for window in windows.values()
    })
    group_a = set(groups[::2])
    group_b = set(groups[1::2])
    ids_a = {
        keyframe_id
        for keyframe_id, window in windows.items()
        if window["group"] in group_a
    }
    ids_b = set(windows) - ids_a
    return ids_a, ids_b

GROUP_A_IDS, GROUP_B_IDS = group_split_ids(valid_windows)
unique_validation_groups = {
    window["group"]
    for window in valid_windows.values()
}
assert len(unique_validation_groups) >= 2, (
    "Need at least two validation sequences/groups for the A/B consistency check. "
    "Inspect sequence grouping before continuing."
)
assert GROUP_A_IDS and GROUP_B_IDS
print("Validation groups:", len(unique_validation_groups))
print("Group A keyframes:", len(GROUP_A_IDS))
print("Group B keyframes:", len(GROUP_B_IDS))

validation_candidate_records = {}
validation_rows = []

if RUN_VALID_CANDIDATE_SWEEP:
    candidate_names = list(CANDIDATE_CONFIGS)
    if ENABLE_TRUE_PROPAGATION:
        candidate_names.append("rerank_l35_d15_propagation")

    for candidate_name in tqdm(
        candidate_names,
        desc="evaluate temporal candidates",
    ):
        if candidate_name == "rerank_l35_d15_propagation":
            base_name = "rerank_l35_d15"
            internal_records = generate_candidate_records(
                base_name,
                valid_windows,
                valid_cache,
                valid_tracks,
                include_propagation=True,
            )
        else:
            internal_records = generate_candidate_records(
                candidate_name,
                valid_windows,
                valid_cache,
                valid_tracks,
                include_propagation=False,
            )

        clean_records = strip_internal_fields(internal_records)
        validation_candidate_records[candidate_name] = clean_records

        coco_metrics = evaluate_coco_predictions(clean_records)
        hungarian_metrics = evaluate_hungarian_map(clean_records)
        group_a_metrics = evaluate_coco_predictions(
            clean_records,
            image_ids=GROUP_A_IDS,
        )
        group_b_metrics = evaluate_coco_predictions(
            clean_records,
            image_ids=GROUP_B_IDS,
        )

        validation_rows.append({
            "candidate": candidate_name,
            **coco_metrics,
            **hungarian_metrics,
            "group_a_mAP50_95": group_a_metrics["coco_mAP50_95"],
            "group_b_mAP50_95": group_b_metrics["coco_mAP50_95"],
            "detections": len(clean_records),
        })

    validation_results_df = pd.DataFrame(validation_rows)
    validation_results_df["coco_delta"] = (
        validation_results_df["coco_mAP50_95"]
        - baseline_coco_metrics["coco_mAP50_95"]
    )
    validation_results_df["hungarian_delta"] = (
        validation_results_df["hungarian_mAP50_95"]
        - baseline_hungarian_metrics["hungarian_mAP50_95"]
    )
    validation_results_df["group_min_delta"] = np.minimum(
        validation_results_df["group_a_mAP50_95"]
        - evaluate_coco_predictions(
            baseline_val_records,
            image_ids=GROUP_A_IDS,
        )["coco_mAP50_95"],
        validation_results_df["group_b_mAP50_95"]
        - evaluate_coco_predictions(
            baseline_val_records,
            image_ids=GROUP_B_IDS,
        )["coco_mAP50_95"],
    )

    validation_results_df = validation_results_df.sort_values(
        [
            "coco_mAP50_95",
            "hungarian_mAP50_95",
            "group_min_delta",
        ],
        ascending=False,
    ).reset_index(drop=True)

    validation_results_df.to_csv(
        REPORT_DIR / "temporal_candidate_validation_results.csv",
        index=False,
    )

    display_columns = [
        "candidate",
        "coco_mAP50_95",
        "coco_delta",
        "hungarian_mAP50_95",
        "hungarian_delta",
        "coco_mAP50",
        "coco_mAP75",
        "coco_mAR100",
        "coco_s_map",
        "AP_bee",
        "AP_bumblebee",
        "AP_hoverfly",
        "AP_moth",
        "group_a_mAP50_95",
        "group_b_mAP50_95",
        "group_min_delta",
        "detections",
    ]
    display(
        validation_results_df[display_columns].style.format({
            column: "{:.4f}"
            for column in display_columns
            if column not in {"candidate", "detections"}
        })
    )
else:
    validation_results_df = pd.read_csv(
        REPORT_DIR / "temporal_candidate_validation_results.csv"
    )

BEST_CANDIDATE = str(validation_results_df.iloc[0]["candidate"])
print("Top validation candidate:", BEST_CANDIDATE)

Validation groups: 21
Group A keyframes: 462
Group B keyframes: 470


evaluate temporal candidates:   0%|          | 0/13 [00:00<?, ?it/s]

,candidate,coco_mAP50_95,coco_delta,hungarian_mAP50_95,hungarian_delta,coco_mAP50,coco_mAP75,coco_mAR100,coco_s_map,AP_bee,AP_bumblebee,AP_hoverfly,AP_moth,group_a_mAP50_95,group_b_mAP50_95,group_min_delta,detections
0,baseline,0.8866,0.0000,0.8340,0.0000,0.9924,0.9602,0.9040,0.7088,0.8540,0.9614,0.7813,0.9496,0.8824,0.8961,0.0000,17637
1,seq_bounded_max,0.8863,-0.0002,0.8255,-0.0085,0.9925,0.9602,0.9045,0.7089,0.8530,0.9614,0.7819,0.9491,0.8822,0.8962,-0.0002,36174
2,rerank_l25_d00,0.8843,-0.0023,0.8215,-0.0125,0.9924,0.9598,0.9048,0.7055,0.8514,0.9609,0.7781,0.9469,0.8789,0.8954,-0.0035,73627
3,rerank_l25_d10,0.8843,-0.0023,0.8225,-0.0116,0.9924,0.9597,0.9047,0.7055,0.8513,0.9609,0.7780,0.9469,0.8789,0.8954,-0.0035,64854
4,rerank_l25_d15,0.8842,-0.0024,0.8236,-0.0105,0.9924,0.9597,0.9047,0.7055,0.8513,0.9609,0.7778,0.9468,0.8789,0.8954,-0.0035,60380
5,rerank_l35_d00,0.8832,-0.0034,0.8184,-0.0157,0.9924,0.9593,0.9049,0.7041,0.8490,0.9612,0.7769,0.9457,0.8777,0.8948,-0.0047,89690
6,rerank_l35_d10,0.8831,-0.0034,0.8194,-0.0146,0.9924,0.9592,0.9048,0.7041,0.8489,0.9612,0.7768,0.9457,0.8776,0.8947,-0.0048,86224
7,rerank_l35_d15,0.8831,-0.0035,0.8196,-0.0145,0.9924,0.9592,0.9048,0.7041,0.8488,0.9612,0.7767,0.9458,0.8775,0.8947,-0.0049,83782
8,rerank_l35_d15_propagation,0.8831,-0.0035,0.8196,-0.0145,0.9924,0.9592,0.9048,0.7041,0.8488,0.9612,0.7767,0.9458,0.8775,0.8947,-0.0049,83782
9,rerank_l45_d00,0.8807,-0.0059,0.8100,-0.0240,0.9924,0.9583,0.9051,0.6979,0.8457,0.9614,0.7736,0.9422,0.8751,0.8935,-0.0073,92880


Top validation candidate: baseline


## 17. Visual inspection of the strongest candidate

The notebook saves side-by-side baseline/candidate overlays for validation images with the largest prediction-count changes.

In [20]:
PALETTE = {
    1: "yellow",
    2: "cyan",
    3: "magenta",
    4: "orange",
}

def group_records_by_image(records):
    grouped = defaultdict(list)
    for record in records:
        grouped[int(record["image_id"])].append(record)
    return grouped

def draw_records(image_path, records, title):
    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)
    draw.rectangle((0, 0, 900, 32), fill="black")
    draw.text((8, 8), title, fill="white")

    for record in sorted(
        records,
        key=lambda item: float(item["score"]),
        reverse=True,
    )[:FINAL_MAX_DETECTIONS]:
        x, y, width, height = record["bbox"]
        category_id = int(record["category_id"])
        color = PALETTE.get(category_id, "white")
        draw.rectangle(
            (x, y, x + width, y + height),
            outline=color,
            width=2,
        )
        draw.text(
            (x, max(34, y - 12)),
            f"{CLASS_NAMES[category_id - 1]} {record['score']:.2f}",
            fill=color,
        )
    return image

if BEST_CANDIDATE in validation_candidate_records:
    baseline_grouped = group_records_by_image(baseline_val_records)
    candidate_grouped = group_records_by_image(
        validation_candidate_records[BEST_CANDIDATE]
    )

    changes = []
    for image_id in valid_windows:
        baseline_count = len(baseline_grouped.get(image_id, []))
        candidate_count = len(candidate_grouped.get(image_id, []))
        changes.append((
            abs(candidate_count - baseline_count),
            image_id,
        ))

    output_dir = DEBUG_DIR / BEST_CANDIDATE
    output_dir.mkdir(parents=True, exist_ok=True)

    for _, image_id in sorted(changes, reverse=True)[:12]:
        left = draw_records(
            valid_frame_paths[image_id],
            baseline_grouped.get(image_id, []),
            "baseline",
        )
        right = draw_records(
            valid_frame_paths[image_id],
            candidate_grouped.get(image_id, []),
            BEST_CANDIDATE,
        )
        canvas = Image.new(
            "RGB",
            (left.width + right.width, max(left.height, right.height)),
        )
        canvas.paste(left, (0, 0))
        canvas.paste(right, (left.width, 0))
        canvas.save(output_dir / f"{image_id}.jpg", quality=90)

    print("Saved debug comparisons:", output_dir)

Saved debug comparisons: /content/drive/MyDrive/BuzzSpot/temporal_rfdetr_1344/debug_visuals/baseline


## 18. Export the strongest test candidates

The notebook exports the top validation-ranked non-baseline candidates as separate valid submission zips. It does not upload anything automatically.

In [21]:
def write_submission(records, prediction_path, zip_path):
    prediction_path = Path(prediction_path)
    zip_path = Path(zip_path)
    clean_records = strip_internal_fields(records)
    prediction_path.write_text(json.dumps(clean_records))

    with zipfile.ZipFile(
        zip_path,
        "w",
        zipfile.ZIP_DEFLATED,
    ) as archive:
        archive.write(
            prediction_path,
            arcname="predictions.json",
        )

def validate_submission(prediction_path, zip_path):
    records = json.loads(Path(prediction_path).read_text())
    valid_image_ids = set(test_windows)

    with zipfile.ZipFile(zip_path, "r") as archive:
        assert archive.namelist() == ["predictions.json"]

    counts = Counter()
    for record in records:
        image_id = int(record["image_id"])
        category_id = int(record["category_id"])
        score = float(record["score"])
        x, y, width, height = map(float, record["bbox"])

        assert image_id in valid_image_ids
        assert category_id in {1, 2, 3, 4}
        assert 0.0 <= score <= 1.0
        assert width > 0 and height > 0
        assert x >= 0 and y >= 0
        assert x + width <= W_FULL + 1e-6
        assert y + height <= H_FULL + 1e-6
        counts[image_id] += 1

    assert not counts or max(counts.values()) <= FINAL_MAX_DETECTIONS

    print("Submission validation passed:")
    print(" detections:", len(records))
    print(" predicted images:", len(counts), "/", len(valid_image_ids))
    print(" max detections/image:", max(counts.values()) if counts else 0)

if RUN_TEST_EXPORT:
    ranked_nonbaseline = [
        candidate
        for candidate in validation_results_df["candidate"].tolist()
        if candidate != "baseline"
    ]
    export_candidates = ranked_nonbaseline[:AUTO_EXPORT_TOP_K]

    print("Exporting test candidates:", export_candidates)
    exported = []

    for candidate_name in export_candidates:
        if candidate_name == "rerank_l35_d15_propagation":
            internal_records = generate_candidate_records(
                "rerank_l35_d15",
                test_windows,
                test_cache,
                test_tracks,
                include_propagation=True,
            )
        else:
            internal_records = generate_candidate_records(
                candidate_name,
                test_windows,
                test_cache,
                test_tracks,
                include_propagation=False,
            )

        safe_name = re.sub(r"[^a-zA-Z0-9_]+", "_", candidate_name)
        prediction_path = (
            SUBMISSIONS_DIR
            / f"predictions_rfdetr_large_1344_temporal_{safe_name}_conf010_top100.json"
        )
        zip_path = (
            SUBMISSIONS_DIR
            / f"submission_rfdetr_large_1344_temporal_{safe_name}_conf010_top100.zip"
        )

        write_submission(
            internal_records,
            prediction_path,
            zip_path,
        )
        validate_submission(
            prediction_path,
            zip_path,
        )

        exported.append({
            "candidate": candidate_name,
            "prediction_path": str(prediction_path),
            "zip_path": str(zip_path),
            "detections": len(internal_records),
        })

    export_df = pd.DataFrame(exported)
    export_df.to_csv(
        REPORT_DIR / "exported_temporal_submissions.csv",
        index=False,
    )
    display(export_df)
else:
    print("RUN_TEST_EXPORT=False -> no test zips created.")

Exporting test candidates: ['seq_bounded_max', 'rerank_l25_d00', 'rerank_l25_d10']
Submission validation passed:
 detections: 337830
 predicted images: 4763 / 4763
 max detections/image: 100
Submission validation passed:
 detections: 451254
 predicted images: 4763 / 4763
 max detections/image: 100
Submission validation passed:
 detections: 435662
 predicted images: 4763 / 4763
 max detections/image: 100


,candidate,prediction_path,zip_path,detections
0,seq_bounded_max,/content/drive/MyDrive/BuzzSpot/submissions/pr...,/content/drive/MyDrive/BuzzSpot/submissions/su...,337830
1,rerank_l25_d00,/content/drive/MyDrive/BuzzSpot/submissions/pr...,/content/drive/MyDrive/BuzzSpot/submissions/su...,451254
2,rerank_l25_d10,/content/drive/MyDrive/BuzzSpot/submissions/pr...,/content/drive/MyDrive/BuzzSpot/submissions/su...,435662


## 19. Final report

Inspect these artifacts before using the remaining leaderboard submission:

- `reports/hoverfly_rescueability_summary.json`
- `reports/temporal_candidate_validation_results.csv`
- `debug_visuals/<best-candidate>/`
- `reports/exported_temporal_submissions.csv`

The banked standalone score remains untouched. Upload only the temporal zip whose validation behavior, group consistency, class metrics, and visual changes you accept.

In [22]:
print("Temporal workspace:", TEMPORAL_DIR)
print("Validation ranking:", REPORT_DIR / "temporal_candidate_validation_results.csv")
print("Hoverfly diagnostic:", REPORT_DIR / "hoverfly_rescueability_summary.json")
print("Export manifest:", REPORT_DIR / "exported_temporal_submissions.csv")
print("Debug visuals:", DEBUG_DIR)
print()
print("Baseline checkpoint:", SOURCE_WEIGHTS)
print("Baseline hidden score already banked: 0.404906326153285")
print("No training was performed by this notebook.")

Temporal workspace: /content/drive/MyDrive/BuzzSpot/temporal_rfdetr_1344
Validation ranking: /content/drive/MyDrive/BuzzSpot/temporal_rfdetr_1344/reports/temporal_candidate_validation_results.csv
Hoverfly diagnostic: /content/drive/MyDrive/BuzzSpot/temporal_rfdetr_1344/reports/hoverfly_rescueability_summary.json
Export manifest: /content/drive/MyDrive/BuzzSpot/temporal_rfdetr_1344/reports/exported_temporal_submissions.csv
Debug visuals: /content/drive/MyDrive/BuzzSpot/temporal_rfdetr_1344/debug_visuals

Baseline checkpoint: /content/drive/MyDrive/BuzzSpot/weights/rfdetr_large_trainval_os_1344_32ep_selected_for_inference.pth
Baseline hidden score already banked: 0.404906326153285
No training was performed by this notebook.
